# ตอนที่ 7: Model Distillation — คำตอบที่ผิดของครู คือส่วนที่มีค่าที่สุด

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kobkrit/thai-llm-tutorials/blob/main/notebooks/07_model_distillation.ipynb)

โน้ตบุ๊กประกอบบทความ **[LLM 7/10] Model Distillation: คำตอบที่ผิดของครู คือส่วนที่มีค่าที่สุด**

โดย **ดร.กอบกฤตย์ วิริยะยุทธกร** — ผู้สร้าง OpenThaiGPT, CEO บริษัท iApp Technology
บทความฉบับเต็ม: [kobkrit.com](https://kobkrit.com/blog/llm-07-model-distillation)

---

## โครงของโน้ตบุ๊กนี้ (เรียงตรงกับหัวข้อในบทความ)

1. ปัญหา (Problem statement)
2. เราจะทำอะไร (Solution)
3. สมการ (Equation) — รวมการอนุมานที่มาของตัวคูณ $T^2$
4. เห็นภาพสมการ (Visualize)
5. เตรียมสภาพแวดล้อม (Environment) — โหลดครูกับนักเรียนพร้อมกัน แล้ววัด baseline
6. เตรียมข้อมูล (Data) — เฉลยจริง, คำตอบครู และ top-64 logits ของครู
7. โค้ดหลัก (Main code) — **เขียน KD loss เองด้วยมือ แล้วพิสูจน์กับ float64 ที่เขียนแยกอิสระ**
8. ผลลัพธ์ (Results)
9. เปรียบเทียบ (Comparison) — gap closed % และสิ่งที่นักเรียนรับติดมาจากครู
10. สรุป (Summary)

---

## หัวใจของโน้ตบุ๊กนี้

หัวข้อที่ 7 มีบรรทัดนี้อยู่:

```python
assert diff < 1e-6
```

`diff` คือผลต่างระหว่าง KD loss ที่เราเขียนเองด้วย PyTorch กับ KD loss สูตรเดียวกัน
ที่เขียน **แยกอิสระอีกครั้งด้วย float64 + `math.exp` ล้วน ๆ** (ไม่มี `F.softmax` แม้แต่ตัวเดียว)
บน tensor เดียวกันเป๊ะ ถ้าสองทางที่เขียนคนละสไตล์ให้ค่าตรงกันถึงทศนิยมตำแหน่งที่หก
แปลว่า masking, การ shift ตำแหน่ง และการ renormalize บน top-64 ถูกต้องตรงกันจริง
ไม่ใช่บังเอิญรันผ่าน — โน้ตบุ๊กนี้พิสูจน์ตัวเอง ไม่ได้ขอให้เชื่อ

และยังมี assert ชุดที่สอง: กวาดค่า $T$ แล้ววัด gradient จริง เพื่อยืนยันว่า
**ถ้าไม่คูณ $T^2$ คืน gradient ของ soft loss จะไหลลงตาม $1/T^2$**
ซึ่งเท่ากับแอบหาร learning rate ของตัวเองโดยไม่รู้ตัว

---

## ก่อนเริ่ม

- โน้ตบุ๊กนี้ออกแบบมาให้รันได้บน **Google Colab แบบฟรี (Tesla T4)**
  เลือก `Runtime > Change runtime type > T4 GPU` ก่อนรันเซลล์แรก
- T4 เป็นสถาปัตยกรรม Turing (sm_75) ซึ่ง **ไม่รองรับ bfloat16 และไม่รองรับ FlashAttention-2**
  เราจึงกำหนด `torch_dtype=torch.float16`, `attn_implementation="sdpa"` และ `fp16=True` เอง
  อย่างชัดเจนทุกครั้ง (รายละเอียดอยู่ในคอมเมนต์ของ Cell 0 — อ่านให้ครบ)
- ตอนนี้ VRAM ตึงเป็นพิเศษ เพราะ **ครู (1.7B) กับนักเรียน (0.6B) ต้องอยู่บนการ์ดพร้อมกัน**
  ในครึ่งแรกของโน้ตบุ๊ก — เราจะพิมพ์ตัวเลข VRAM ให้ดูทุกจุดสำคัญ
- ทุกตอนในซีรีส์นี้ใช้ชุดวัดผลกลางตัวเดียวกันคือ `kobeval` และเบนช์มาร์ก **KobEval-TH**
  (120 ข้อ, 4 slice) เพื่อให้ตัวเลขของแต่ละตอนเทียบกันได้จริง
- **ห้ามเทรนบน KobEval-TH เด็ดขาด** ชุดนี้เป็นชุดวัดผลอย่างเดียว
  ข้อมูลที่ใช้เทรนในตอนนี้มาจาก `airesearch/wangchanx-seed-free-synthetic-instruct-thai-120k`
  ซึ่งแยกขาดจากชุดวัดผล

## เวลาที่ใช้โดยประมาณบน T4 ฟรี (รวมเวลาโหลดโมเดล)

| ขั้นตอน | เวลาโดยประมาณ |
|---|---|
| Cell 0 ติดตั้ง dependencies | ~2 นาที |
| Cell 1 โหลดครู+นักเรียน + วัด baseline นักเรียน + วัดเพดานครู (TH-INSTR) | ~6 นาที |
| เตรียมข้อมูล + ครู generate คำตอบ (SeqKD) | ~4 นาที |
| เก็บ top-64 logits ของครู + tokens.json | ~2 นาที |
| เทรน SeqKD + เทรน logit KD (24 step ต่อรอบ) | ~3 นาที |
| วัดผลซ้ำ 2 รอบ + TH-SAFE + กราฟ + export | ~6 นาที |
| **รวม** | **~20-23 นาที** |

ถ้า T4 ที่ได้ช้ากว่าปกติ ให้ลด `N_TRAIN` ในหัวข้อที่ 6 (เช่นเหลือ 256)
— เรื่องที่เล่าจะเหมือนเดิมทุกประการ เพียงแต่ขนาดของผลจะเล็กลง


In [ ]:
# =============================================================================
# CELL 0 -- SETUP
# ชุดนี้ต้อง "เหมือนกันทุกตัวอักษร" (byte-identical) ในโน้ตบุ๊กทั้ง 10 ตอน
# ตรวจสอบด้วย: python3 scripts/check_cell0.py
# =============================================================================
# ทำไมต้องเหมือนกันทุกตอน: ถ้า setup ต่างกันแม้นิดเดียว ตัวเลขที่วัดได้จาก
# แต่ละตอนจะเปรียบเทียบกันไม่ได้ ซึ่งทำให้ทั้งซีรีส์นี้ไร้ความหมาย
# -----------------------------------------------------------------------------

# 1) ติดตั้ง dependencies (pin เวอร์ชันไว้ทั้งหมด เพื่อให้ผลลัพธ์ทำซ้ำได้)
#    หมายเหตุ: เราจงใจ "ไม่" ติดตั้ง torch ใหม่ เพราะ Colab มี torch ที่ build
#    มาให้ตรงกับ CUDA driver อยู่แล้ว การ pip install torch ทับคือสาเหตุอันดับ
#    หนึ่งที่ทำให้ notebook พังบน Colab
!pip install -q \
    "transformers==4.51.3" \
    "accelerate==1.6.0" \
    "datasets==3.5.0" \
    "peft==0.15.1" \
    "trl==0.16.1" \
    "bitsandbytes==0.45.5" \
    "sentencepiece==0.2.0" \
    "matplotlib==3.10.1"

# 2) ติดตั้งฟอนต์ไทยให้ matplotlib (ไม่งั้นกราฟจะขึ้นเป็นสี่เหลี่ยม tofu)
!apt-get install -y fonts-thai-tlwg > /dev/null 2>&1

# 3) ดึง repo (ได้ทั้งแพ็กเกจ kobeval และชุดข้อมูล KobEval-TH)
import os
REPO_DIR = "/content/thai-llm-tutorials"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/kobkrit/thai-llm-tutorials.git /content/thai-llm-tutorials
!pip install -q -e /content/thai-llm-tutorials

# ลดการแตกกระจายของหน่วยความจำ (fragmentation) -- ต้องตั้งก่อน torch แตะ CUDA
# T4 มี VRAM จำกัด พอ tensor ก้อนใหญ่ถูกจอง/คืนสลับกัน จะเกิดช่องว่างที่ใช้ต่อไม่ได้
# จน OOM ทั้งที่ยังเหลือที่รวม ๆ อยู่ (ข้อความ error ของ PyTorch เองก็แนะนำค่านี้)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import random
import numpy as np
import torch

# 4) ยืนยันว่ามี GPU จริง -- ถ้าไม่มีให้หยุดตรงนี้เลย ดีกว่าไปพังตอน train
assert torch.cuda.is_available(), (
    "ไม่พบ GPU! ไปที่ Runtime > Change runtime type > Hardware accelerator > GPU "
    "แล้วรันเซลล์นี้ใหม่"
)

GPU_NAME = torch.cuda.get_device_name(0)
CAPABILITY = torch.cuda.get_device_capability(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
# bf16 "แบบมีฮาร์ดแวร์รองรับจริง" เริ่มที่ Ampere (sm_80) ขึ้นไปเท่านั้น
#
# หมายเหตุสำคัญ: torch.cuda.is_bf16_supported() ตอบ True บน T4 (sm_75) ด้วย
# ตั้งแต่ torch รุ่นใหม่ ๆ เพราะมันนับ "การจำลอง (emulation)" ว่ารองรับด้วย
# ซึ่งช้ากว่า fp16 มากและไม่ใช่สิ่งที่เราต้องการ เราจึงเช็ค compute capability
# ตรง ๆ แทน -- นี่คือกับดักจริงที่เจอตอนรันบน Colab
NATIVE_BF16 = CAPABILITY[0] >= 8
SUPPORTS_BF16 = NATIVE_BF16          # ใช้ชื่อเดิมต่อได้ แต่ความหมายคือ "native"

print(f"GPU            : {GPU_NAME}")
print(f"Compute cap.   : sm_{CAPABILITY[0]}{CAPABILITY[1]}")
print(f"VRAM           : {VRAM_GB:.1f} GB")
print(f"SUPPORTS_BF16  : {SUPPORTS_BF16}   (native; torch บอกว่า "
      f"{torch.cuda.is_bf16_supported()} เพราะนับ emulation ด้วย)")
print(f"torch          : {torch.__version__}")

# -----------------------------------------------------------------------------
# 5) จุดสำคัญที่สุดของเซลล์นี้ -- อ่านให้ครบ
#
# ถ้าคุณได้ Tesla T4 (ซึ่งเป็นค่าปกติของ Colab ฟรี) คุณจะเห็น
#     SUPPORTS_BF16 : False   (native)
# ทั้งที่ torch.cuda.is_bf16_supported() ตอบ True -- อย่าเชื่อค่านั้น
#
# T4 เป็นสถาปัตยกรรม Turing (sm_75) ซึ่ง "ไม่รองรับ" สองอย่างนี้:
#   - bfloat16  -> ต้องใช้ float16 แทน
#   - FlashAttention-2 -> ต้องใช้ sdpa แทน
#
# กับดักที่คนเจอบ่อยที่สุด: config ของ Qwen3-0.6B ระบุ torch_dtype: bfloat16
# ไว้ในไฟล์ ดังนั้นถ้าเขียน torch_dtype="auto" transformers จะเชื่อ config
# แล้วโหลดเป็น bf16 บนการ์ดที่ไม่รองรับ ผลคือได้ NaN, loss ไม่ลด หรือ
# โมเดลพ่นข้อความมั่ว โดยไม่มี error ฟ้องให้เห็นเลย
#
# เราจึงกำหนดค่าพวกนี้เอง "อย่างชัดเจน" ทุกครั้ง ไม่พึ่ง auto
# -----------------------------------------------------------------------------
DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16
ATTN_IMPL = "sdpa"          # ห้ามใช้ flash_attention_2 บน T4
USE_FP16 = not SUPPORTS_BF16  # ส่งเข้า TrainingArguments: fp16=USE_FP16, bf16=SUPPORTS_BF16

print(f"\n--> DTYPE      : {DTYPE}")
print(f"--> ATTN_IMPL  : {ATTN_IMPL}")
print(f"--> fp16={USE_FP16}, bf16={SUPPORTS_BF16}  (ใช้คู่นี้กับ TrainingArguments)")

# 6) ตั้ง seed ทุกตัวที่เกี่ยวข้อง เพื่อให้ผลลัพธ์ทำซ้ำได้
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# 7) import kobeval -- ชุดวัดผลกลางที่ใช้เหมือนกันทั้ง 10 ตอน
#    pip install -e ลงทะเบียนแพ็กเกจไว้แล้ว แต่ sys.path ของเคอร์เนลที่รันอยู่
#    อาจยังไม่เห็น จึงใส่ path ของ repo เข้าไปตรง ๆ กันพลาด (idempotent)
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from kobeval import evaluate, compare, plot_before_after, th_ratio, wilson_ci
from kobeval import EVAL_CONTRACT, verify_checksums
from kobeval.plotting import use_thai_font

# 8) ตั้งฟอนต์ไทยให้ matplotlib "ครั้งเดียวตรงนี้" มีผลกับทุกกราฟในโน้ตบุ๊ก
#    ทำไมต้องมีขั้นตอนนี้: apt ติดตั้งฟอนต์ใน (2) หลังจาก matplotlib สร้าง
#    แคชรายชื่อฟอนต์ไปแล้ว มันจึงมองไม่เห็นฟอนต์ไทย และวาดออกมาเป็นกล่อง []
#    use_thai_font() จะสแกนดิสก์ใหม่แล้วลงทะเบียนฟอนต์ให้
_thai_font = use_thai_font()
print(f"Thai font      : {_thai_font or 'ไม่พบ! กราฟภาษาไทยจะเป็นกล่อง [] -- รัน apt-get ในข้อ (2) ใหม่'}")

print(f"\nkobeval contract: {EVAL_CONTRACT}")
print(f"KobEval-TH checksum ok: {verify_checksums()['ok']}")


---

## 1. ปัญหา (Problem statement)

สมมติว่าคุณเดินทางมาครบหกตอน จนได้โมเดลระดับ Qwen3-1.7B ที่ตอบโจทย์ภาษาไทยได้น่าพอใจ
แล้ววันหนึ่งฝ่าย infrastructure ถามคำถามเดียวที่โมเดลตอบไม่ได้: *"ค่า GPU เดือนละเท่าไหร่"*

โมเดล 1.7B กิน VRAM เกือบ 3 เท่าของ 0.6B และตอบช้ากว่าราว 2.5 เท่า
ที่ 100 คำขอต่อวินาที ส่วนต่างนี้คือจำนวนการ์ดที่ต้องซื้อเพิ่มทุกเดือนตลอดอายุระบบ

| ทางเลือก | คุณภาพ | ต้นทุนตอนใช้งาน |
|---|---|---|
| Deploy ครู 1.7B ตรง ๆ | ดีที่สุด | VRAM ~3 เท่า ช้ากว่า ~2.5 เท่า |
| Deploy นักเรียน 0.6B ตรง ๆ | ตกลงชัดเจน | ถูกและเร็ว |
| SFT นักเรียนด้วยเฉลยจริง | ดีขึ้นบ้าง | ถูกและเร็ว |
| **Model distillation** | ขยับเข้าหาครู | ถูกและเร็ว **เท่านักเรียนเป๊ะ** |

คำถามคือ แถวสุดท้ายรู้อะไรที่แถว SFT ไม่รู้ — เทรนเหมือนกัน ข้อมูลชุดเดียวกัน ต่างกันตรงไหน

คำตอบอยู่ที่ **ปริมาณข้อมูลต่อหนึ่ง token** ฉลากแข็ง (hard label) คือ one-hot vector:
บอกแค่ว่า "คำตอบคือ 3" จบ แต่ distribution ของครูที่ถูกทำให้นุ่มด้วย temperature บอกว่า
*"คำตอบคือ 3 — แต่ 8 ก็พอเป็นไปได้อยู่ ส่วน 'แมว' นี่ไร้สาระสิ้นเชิง"*
การจัดอันดับเหนือ **คำตอบที่ผิดทั้งหมด** นี่แหละที่ Hinton เรียกว่า **dark knowledge**
มันเข้ารหัสโครงสร้างความคล้ายของโลกเอาไว้ และหายไปหมดเกลี้ยงเมื่อคุณเก็บแต่ argmax

ทุกตำแหน่ง token ครูมีตัวเลขให้ 151,936 ตัว (ขนาด vocab ของ Qwen3) — ฉลากแข็งเก็บมาแค่ตัวเดียว

---

## 2. เราจะทำอะไร (Solution)

เราจะเอา **`Qwen/Qwen3-1.7B`** เป็นครู และ **`Qwen/Qwen3-0.6B-Base`** เป็นนักเรียน
แล้วกลั่นด้วยกัน 2 ระดับหลัก + 1 ทางเลือกเสริม ซึ่งส่งข้อมูลจากครูละเอียดขึ้นเรื่อย ๆ:

1. **SeqKD** — ให้ครู generate คำตอบ แล้ว SFT นักเรียนบนคำตอบนั้น (ตัวอย่างจาก distribution)
2. **Logit KD** — ให้นักเรียนเลียนแบบ distribution ของครูทีละตำแหน่ง token (ตัว distribution เอง)
   ผ่าน **top-64 logits ที่ precompute แบบ offline**
3. **GKD** (ทางเลือกเสริม ข้ามได้) — นักเรียนสุ่มคำตอบเอง แล้วครูให้คะแนนแบบ on-policy

ทั้งสอง regime หลักเทรนด้วย LoRA เดียวกัน จำนวน step เท่ากัน hyperparameter เดียวกัน
ต่างกันแค่ **สัญญาณที่ใช้สอน** แล้ววัดทุกอย่างด้วย KobEval-TH ชุดเดิมของซีรีส์
สรุปเป็นตัวเลขเดียวคือ **gap closed %** — ช่องว่างระหว่างนักเรียนกับครู ปิดไปแล้วกี่เปอร์เซ็นต์

> **แนวคิดหลักของตอนนี้:** ฉลากแข็งบอกว่า "คำตอบคือ 3" — distribution ของครูบอกว่า
> "3 แต่ 8 ก็เกือบใช่ และ 'แมว' เป็นไปไม่ได้" การจัดอันดับเหนือคำตอบที่ผิดคือ dark knowledge
> และ **temperature คือปุ่มที่เปิดเผยมัน** ที่ $T=1$ ความรู้นี้ถูกบีบจนมองไม่เห็น
> พอยก $T$ ขึ้น มันจึงกลายเป็น signal ที่เทรนได้จริง

วิธีนี้ไม่ใช่เทคนิคห้องทดลอง — Qwen3-0.6B ที่เราใช้กันมาทั้งซีรีส์ ก็ถูกฝึกด้วย
strong-to-weak distillation จากรุ่นใหญ่ของตระกูลมันเอง ตอนนี้เราทำสิ่งเดียวกัน
ในขนาดที่ Colab ฟรีรับไหว


---

## 3. สมการ (Equation)

### 3.1 KD loss ของ Hinton

$$
\mathcal{L}_{\text{KD}} = (1-\alpha)\,\mathcal{L}_{\text{CE}}\big(y,\ \text{softmax}(z_S)\big) + \alpha\, T^2\,\mathbb{D}_{\text{KL}}\Big(\text{softmax}(z_T/T)\ \big\|\ \text{softmax}(z_S/T)\Big)
$$

- $z_S, z_T$ = logits ของนักเรียนและครู ที่ตำแหน่ง token เดียวกัน บน input เดียวกัน
- $y$ = เฉลยจริง (hard label) — พจน์แรกคือ cross-entropy ปกติ เหมือน SFT ทุกประการ
- $T$ = temperature หารเข้าไปใน logits **ทั้งสองฝั่ง** ก่อน softmax (เราใช้ $T=2$)
- $\alpha$ = น้ำหนักของพจน์ soft (เราใช้ $0.9$ — ฟังครูเป็นหลัก มีเฉลยจริงคอยกันหลุด)

สังเกตทิศของ KL: ครูอยู่หน้า นี่คือ **forward KL** ที่บังคับนักเรียนให้แผ่ความน่าจะเป็น
ครอบคลุมทุกที่ที่ครูให้น้ำหนัก

### 3.2 อนุมานที่มาของ $T^2$ — สองบรรทัดที่แยก "เข้าใจ" ออกจาก "ก๊อปมา"

**บรรทัดที่ 1** — gradient ของพจน์ soft ต่อ logit ของนักเรียนหนึ่งตัว คือ gradient มาตรฐานของ
softmax-CE บวก chain rule ผ่าน $z/T$ ซึ่งคาย $1/T$ ออกมาหนึ่งตัว:

$$
\frac{\partial \mathcal{L}_{\text{soft}}}{\partial z_{S,i}} = \frac{1}{T}\Big(q_i^{(T)} - p_i^{(T)}\Big),
\qquad q^{(T)} = \text{softmax}(z_S/T),\quad p^{(T)} = \text{softmax}(z_T/T)
$$

**บรรทัดที่ 2** — เมื่อ $T$ ใหญ่ softmax จะแบนลงรอบ ๆ uniform:
$\text{softmax}(z/T)_i \approx \tfrac{1}{K} + \tfrac{z_i - \bar z}{KT}$ (โดย $K$ = ขนาด vocab)
ดังนั้นผลต่าง $q^{(T)}_i - p^{(T)}_i$ ก็หดลงตาม $1/T$ อีกชั้นหนึ่ง:

$$
\frac{\partial \mathcal{L}_{\text{soft}}}{\partial z_{S,i}} \approx \frac{(z_{S,i} - \bar z_S) - (z_{T,i} - \bar z_T)}{K\,T^2} \;\propto\; \frac{1}{T^2}
$$

gradient ของพจน์ soft สเกลตาม $1/T^2$ ขณะที่พจน์ hard CE ไม่ขึ้นกับ $T$ เลย
**ถ้าไม่คูณ $T^2$ คืน การขยับ $T$ จาก 1 เป็น 4 จะเท่ากับแอบหาร learning rate ของพจน์ soft ด้วย ~16**
คุณจะสรุปว่า "T สูงแล้วไม่เวิร์ก" ทั้งที่จริง ๆ คุณแค่เผลอปิด soft loss ของตัวเองไปเฉย ๆ
หัวข้อ 7 จะ **วัด gradient จริง** เพื่อยืนยันสองบรรทัดนี้ ไม่ใช่ขอให้เชื่อ

ของแถมจากบรรทัดที่ 2: ในลิมิตเดียวกัน soft loss ลดรูปเป็นการ match logits ที่ถูก center แบบ MSE —
KD คือ "logit regression แบบนุ่ม" ที่ให้น้ำหนักส่วนหัวของ distribution มากกว่าส่วนหาง

### 3.3 SeqKD — baseline ราคาถูก (Kim & Rush, 2016)

$$
\mathcal{L}_{\text{SeqKD}} = -\,\mathbb{E}_{\hat y \sim \pi_T}\left[\sum_t \log \pi_S(\hat y_t \mid x, \hat y_{<t})\right]
$$

อ่านแล้วคุ้นไหมครับ — นี่คือ **SFT ธรรมดาบนคำตอบที่ครู generate** ไม่มีอะไรมากกว่านั้น
ข้อดีที่มักถูกมองข้าม: SeqKD **ไม่สนใจว่า tokenizer ตรงกันหรือไม่** เพราะมันส่งข้อความ
ไม่ได้ส่ง logits — โมเดล open-source จำนวนมากที่โฆษณาว่า "distilled from GPT-4"
แท้จริงคือ SeqKD ล้วน ๆ

### 3.4 GKD และ generalized JSD — เส้นที่เชื่อมเรื่องนี้เข้าด้วยกัน

**GKD** (Agarwal et al., 2023) ให้นักเรียนสุ่มคำตอบเอง แล้วครูให้คะแนนบน token เหล่านั้น
พร้อม generalize ระยะทางระหว่าง distribution เป็น

$$
\mathbb{D}^{(\beta)}_{\text{JSD}}(P\,\|\,Q) = \beta\,\mathbb{D}_{\text{KL}}(P\,\|\,M) + (1-\beta)\,\mathbb{D}_{\text{KL}}(Q\,\|\,M),
\qquad M = \beta P + (1-\beta) Q
$$

- $\beta \to 0$ → **forward KL** — mass-covering: นักเรียนต้องแผ่คลุมทุกโหมดของครู (KD คลาสสิกของตอนนี้)
- $\beta \to 1$ → **reverse KL** — mode-seeking: นักเรียนเลือกยึดบางโหมดที่ตัวเองไหว

นักเรียนที่เล็กกว่าครูมาก มักได้ประโยชน์จากฝั่ง mode-seeking
เพราะความจุไม่พอจะคลุมทุกโหมดของครูอยู่แล้ว

---

## 4. เห็นภาพสมการ (Visualize)

รูปของหัวข้อนี้ถูกวาดในเซลล์ **ถัดจาก Cell 1** เพราะกติกาของซีรีส์กำหนดว่า
เซลล์โค้ดที่สองของทุกตอนต้องเป็นการวัด baseline เสมอ ห้ามมีอะไรมาแทรกก่อน

เราจะเอา **logits จริงของครู** (ไม่ใช่เลขสมมติ) มา softmax ที่ $T = 1, 2, 4, 8$
เพื่อดูว่า temperature "เปิดเผย" การจัดอันดับเหนือคำตอบผิดอย่างไร โดยที่ **ไม่ได้เพิ่ม
ข้อมูลอะไรเลย** — logits ชุดเดิมเป๊ะ ๆ มันแค่เปลี่ยนว่าข้อมูลที่มีอยู่แล้วมองเห็นได้แค่ไหน


---

## 5. เตรียมสภาพแวดล้อม (Environment) — และวัด baseline

หลักการของทั้งซีรีส์: **วัดก่อนเสมอ** — แต่ตอนนี้ต้องวัด **สองตัวเลข** ก่อนแตะอะไรทั้งสิ้น:

1. **พื้น** — นักเรียน `Qwen3-0.6B-Base` ก่อนกลั่น (นี่คือ baseline ตามกติกาซีรีส์)
2. **เพดาน** — ครู `Qwen3-1.7B` บนข้อสอบชุดเดียวกัน

เพราะ metric สรุปของตอนนี้คือ gap closed % ซึ่งนิยามจากสองตัวเลขนี้โดยตรง
ถ้าไม่วัดเพดานของครูไว้ก่อน ตัวเลข "นักเรียนดีขึ้น 4 จุด" จะไม่มีความหมายเลย
สังเกตค่า `th_ratio` ของทั้งคู่ด้วย — ครูตอบเป็นภาษาไทยล้วนหรือหลุดอังกฤษ
มีผลตรงต่อสิ่งที่นักเรียนจะรับติดมา

> **หมายเหตุเรื่องเวลา:** เรารัน KobEval-TH เฉพาะ slice `TH-INSTR` (30 ข้อ)
> เพราะการกลั่นในตอนนี้เน้นความสามารถทำตามคำสั่งภาษาไทย ซึ่ง TH-INSTR วัดตรงที่สุด
> ท้ายตอนเราจะรัน `TH-SAFE` เพิ่มบนนักเรียนหลังกลั่น เพื่อดูว่านิสัยของครูติดมาด้วยไหม

### ทำไมต้อง assert ว่า vocab ตรงกัน ก่อนทำ logit KD

KL ในสมการ 3.1 เทียบ distribution สองก้อน **มิติต่อมิติ**: มิติที่ $i$ ของครูต้องหมายถึง
token เดียวกับมิติที่ $i$ ของนักเรียน ถ้า vocab ไม่ตรง คุณกำลังเทียบความน่าจะเป็นของ "กรุงเทพ"
กับ token อื่นที่บังเอิญได้เลขช่องเดียวกัน — ตัวเลขจะไหลออกมาสวยงามและ **ไร้ความหมายสิ้นเชิง**

ซ้ำร้ายกว่านั้น ถ้า tokenizer ต่างกัน ประโยคเดียวกันจะถูกหั่นเป็น token คนละชุด
ตำแหน่งที่ $t$ ของครูกับนักเรียนจึงชี้ไปคนละจุดของข้อความ — เทียบกันไม่ได้ตั้งแต่แกนเวลา
นี่คือเหตุผลที่ logit KD ข้ามตระกูลโมเดล **เป็นไปไม่ได้** (ไม่ใช่แค่ "ยาก")
และทางเดียวที่เหลือคือ SeqKD ซึ่งส่งข้อความ ไม่ส่ง logits

Qwen3 ทั้งตระกูลใช้ tokenizer เดียวกัน เราจึงรอด แต่เราจะ `assert` ให้เห็นกับตา
ไม่ใช่เชื่อจากชื่อรุ่น


In [ ]:
# =============================================================================
# CELL 1 -- BASELINE (วัดผลก่อนเทรน/ก่อนแก้อะไรทั้งสิ้น)
# เซลล์นี้เป็นเซลล์โค้ดที่สองของทุกตอนในซีรีส์
# ตอนนี้วัดสองตัวเลข: "พื้น" (นักเรียนก่อนกลั่น) และ "เพดาน" (ครู)
# =============================================================================
import gc
import json
import math
import time

import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

TEACHER_ID = "Qwen/Qwen3-1.7B"        # ครู: โมเดล instruct
STUDENT_ID = "Qwen/Qwen3-0.6B-Base"   # นักเรียน: โมเดล base (ยังไม่ผ่าน instruction tuning)
KOBEVAL_SLICES = ["TH-INSTR"]
DEV = "cuda:0"

tokenizer = AutoTokenizer.from_pretrained(STUDENT_ID)      # ใช้กับนักเรียนและข้อมูลเทรน
tok_teacher = AutoTokenizer.from_pretrained(TEACHER_ID)    # มี chat template สำหรับครู
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tok_teacher.pad_token is None:
    tok_teacher.pad_token = tok_teacher.eos_token

teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_ID,
    torch_dtype=DTYPE,             # <-- ห้ามใช้ค่า auto เด็ดขาดบน T4 (ดูคอมเมนต์ Cell 0)
    attn_implementation=ATTN_IMPL, # <-- sdpa เท่านั้น
    device_map=DEV,
)
teacher.eval()                     # ครูไม่เรียนอะไรทั้งนั้น
teacher.requires_grad_(False)      # และห้ามมี gradient ไหลเข้าครูเด็ดขาด

student = AutoModelForCausalLM.from_pretrained(
    STUDENT_ID,
    torch_dtype=DTYPE,
    attn_implementation=ATTN_IMPL,
    device_map=DEV,
)
student.eval()

n_t = sum(p.numel() for p in teacher.parameters()) / 1e6
n_s = sum(p.numel() for p in student.parameters()) / 1e6
print(f"ครู     : {TEACHER_ID}  ({n_t:.0f}M params, dtype={next(teacher.parameters()).dtype})")
print(f"นักเรียน: {STUDENT_ID}  ({n_s:.0f}M params, dtype={next(student.parameters()).dtype})")

VRAM_BOTH_GB = torch.cuda.memory_allocated() / (1024 ** 3)
print(f"\nVRAM หลังโหลดสองโมเดลพร้อมกัน: {VRAM_BOTH_GB:.2f} GB "
      f"(จากทั้งหมด {VRAM_GB:.1f} GB)")
print("นี่คืองบที่ตึงที่สุดในซีรีส์ -- ที่เหลือต้องแบ่งให้ activations + logits ชั่วคราว")

# --- assert ที่ต้องมาก่อนทุกอย่าง: vocab ต้องตรงกันทุกช่อง (ดูเหตุผลใน markdown ก่อนหน้า) ---
VOCAB_SIZE = teacher.config.vocab_size
assert teacher.config.vocab_size == student.config.vocab_size, (
    f"vocab_size ไม่ตรงกัน: ครู {teacher.config.vocab_size} vs "
    f"นักเรียน {student.config.vocab_size} -- logit KD เป็นไปไม่ได้ ต้องใช้ SeqKD เท่านั้น"
)
assert tok_teacher.get_vocab() == tokenizer.get_vocab(), (
    "ตาราง token ไม่ตรงกันแม้ขนาดจะเท่ากัน -- มิติที่ i จะหมายถึงคนละ token"
)
print(f"\nvocab ตรงกันทุกช่อง ({VOCAB_SIZE:,} มิติ) -> logit KD เป็นไปได้")

# --- วัดพื้น: นักเรียนก่อนกลั่น (สัญญาการวัดผลตรึงไว้ใน kobeval: greedy,
#     max_new_tokens=256, enable_thinking=False, seed=42) ---
t0 = time.time()
baseline = evaluate(
    student,
    tokenizer,
    slices=KOBEVAL_SLICES,
    seed=42,
    model_name="Qwen3-0.6B-Base (นักเรียน ก่อนกลั่น)",
    out_path="results_baseline.json",
)
print(f"\nใช้เวลาวัดพื้น (นักเรียน): {time.time() - t0:.0f} วินาที")

# --- วัดเพดาน: ครู บนข้อสอบชุดเดียวกัน seed เดียวกัน ---
# compute_ppl=False เพราะครูใช้ chat template ส่วนนักเรียน base ใช้ prompt ดิบ
# PPL ของสองรูปแบบ prompt เทียบกันไม่ได้อยู่แล้ว จึงไม่วัดให้เสียเวลา
t0 = time.time()
teacher_report = evaluate(
    teacher,
    tok_teacher,
    slices=KOBEVAL_SLICES,
    seed=42,
    model_name="Qwen3-1.7B (ครู = เพดาน)",
    compute_ppl=False,
    out_path="results_teacher.json",
)
print(f"ใช้เวลาวัดเพดาน (ครู): {time.time() - t0:.0f} วินาที\n")

for name, rep in (("นักเรียน", baseline), ("ครู", teacher_report)):
    s = rep["slices"]["TH-INSTR"]
    print(f"{name:8s} TH-INSTR acc={s['accuracy']:.3f} "
          f"[95% CI {s['ci_low']:.3f}-{s['ci_high']:.3f}]  "
          f"th_ratio={s['th_ratio']:.2f}  len={s['mean_output_len']:.0f}")

ACC_FLOOR = baseline["slices"]["TH-INSTR"]["accuracy"]
ACC_CEIL = teacher_report["slices"]["TH-INSTR"]["accuracy"]
print(f"\nช่องว่างครู-นักเรียนที่มีให้ปิด: {ACC_CEIL - ACC_FLOOR:+.3f} "
      f"({100 * ACC_CEIL:.1f}% - {100 * ACC_FLOOR:.1f}%)")
print("ทุกอย่างหลังจากนี้จะถูกอ่านเทียบกับพื้นและเพดานคู่นี้")


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 4 -- temperature เปิดเผย dark knowledge (ใช้ logits จริงของครู)
# -----------------------------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np

from kobeval import use_thai_font

use_thai_font()

PROBE_TEXT = "7 × 8 = "     # บริบทสั้น ๆ ที่มีคำตอบถูกหนึ่งเดียว และคำตอบผิดที่ "เกือบใช่"
probe_ids = tok_teacher(PROBE_TEXT, return_tensors="pt").to(DEV)

with torch.no_grad():
    z = teacher(**probe_ids).logits[0, -1].float().cpu()    # [V] logits ตำแหน่งถัดไป

TEMPS = [1.0, 2.0, 4.0, 8.0]
top_val, top_idx = z.topk(8)
tok_labels = [repr(tok_teacher.decode([int(i)]))[1:-1] for i in top_idx]

fig, ax = plt.subplots(figsize=(10.5, 4.4))
width = 0.2
xs = np.arange(len(top_idx))
for j, (T, color) in enumerate(zip(TEMPS, ["#1d4ed8", "#2563eb", "#f59e0b", "#dc2626"])):
    probs_full = torch.softmax(z / T, dim=-1)
    entropy = float(-(probs_full * probs_full.clamp_min(1e-12).log()).sum())
    p_top = probs_full[top_idx]
    ax.bar(xs + (j - 1.5) * width, p_top, width, color=color,
           label=f"T={T:g} (entropy={entropy:.2f} nat)")
    # การจัดอันดับต้องไม่เปลี่ยน -- temperature ไม่ได้เพิ่มหรือสลับข้อมูล
    assert torch.equal(probs_full[top_idx].argsort(descending=True),
                       z[top_idx].argsort(descending=True))

ax.set_xticks(xs)
ax.set_xticklabels(tok_labels, fontsize=9)
ax.set_ylabel("ความน่าจะเป็นหลัง softmax")
ax.set_yscale("log")
ax.set_title("logits ชุดเดิมเป๊ะ ๆ -- T สูงขึ้นแค่ทำให้การจัดอันดับเหนือคำตอบผิดมองเห็นได้",
             fontsize=11)
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig("fig_temperature_softens.png", dpi=150, bbox_inches="tight")
plt.show()

p1 = torch.softmax(z / 1.0, dim=-1)
p2 = torch.softmax(z / 2.0, dim=-1)
best_wrong = int(top_idx[1])
print(f"บันทึกรูปแล้ว: fig_temperature_softens.png")
print(f"ครูตอบ (top-1): {tok_labels[0]!r}  p(T=1)={float(p1[top_idx[0]]):.4f}")
print(f"คำตอบผิดที่ดีที่สุด: {tok_labels[1]!r}  p(T=1)={float(p1[best_wrong]):.5f} "
      f"-> p(T=2)={float(p2[best_wrong]):.5f}")
print("gradient ที่ไหลผ่านคำตอบผิดแปรตาม p ของมัน -- ที่ T=1 จึงแทบเป็นศูนย์")

cov64 = float(p2.topk(64).values.sum())
print(f"\nมวลความน่าจะเป็นของ top-64 ที่ T=2: {100 * cov64:.2f}% "
      f"(หาง {VOCAB_SIZE - 64:,} token ที่เหลือแบ่งกันไม่ถึง {100 * (1 - cov64):.2f}%)")
print("นี่คือเหตุผลเชิงตัวเลขที่หัวข้อ 6 จะเก็บ logits ครูแค่ 64 อันดับแรก")


---

## 6. เตรียมข้อมูล (Data)

เราสุ่ม **3,000 prompt** จาก `airesearch/wangchanx-seed-free-synthetic-instruct-thai-120k`
(ชุด instruction ภาษาไทยที่มีเฉลยอ้างอิงครบ) เป็นคลังตั้งต้น แล้วใช้จริง `N_TRAIN` ข้อแรก
ที่ผ่านการคัดกรองความยาว — บน T4 ฟรีเราตั้งไว้ 384 ข้อ (24 optimizer step ต่อ regime)
ซึ่งพอให้เห็นกลไกโดยไม่เผางบเวลา ถ้ามีเวลาก็เพิ่มได้ตรง ๆ

จากข้อมูลก้อนเดียวนี้ เราสร้างวัตถุดิบ 3 ชิ้น:

| ชิ้น | ใช้กับ regime | สร้างอย่างไร |
|---|---|---|
| **เฉลยจริง** | logit KD (พจน์ hard CE) | ใช้ตรง ๆ จากชุดข้อมูล |
| **คำตอบของครู** | SeqKD | ครู generate แบบ greedy ครั้งเดียว เก็บไว้ |
| **top-64 logits ของครู** | logit KD (พจน์ soft KL) | forward ครูผ่านประโยคเฉลยจริง เก็บ 64 อันดับแรก |

ข้อสังเกตสำคัญ: นักเรียนเป็นโมเดล **base** ที่ไม่มี chat template เราจึงจัดรูปข้อมูลเป็น
ข้อความล้วนแบบ `คำถาม: ...\nคำตอบ: ...` เหมือนตอนที่ 1-2 และ mask ฝั่งคำถามเป็น `-100`
เพื่อไม่ให้ gradient ไปหัดเขียนคำถาม (บทเรียนจากตอนที่ 2)
ครูถูก forward บน **token ชุดเดียวกันเป๊ะ** กับที่นักเรียนเห็นตอนเทรน —
ทำได้เพราะ vocab ตรงกันทุกช่อง ตามที่ assert ไว้ใน Cell 1

### ทำไมต้อง top-64 ไม่เก็บทั้ง vocab

เพราะเก็บทั้ง vocab **ไม่ได้จริง ๆ** — เซลล์ถัดจากเซลล์ข้อมูลจะคิดเลขให้ดูสด ๆ
ว่า logits เต็ม vocab หนึ่ง batch คือ tensor ขนาดหลายร้อย MB และทั้งชุดข้อมูลคือหลายร้อย GB
ขณะที่ top-64 เล็กลงหลายร้อยเท่า โดยทิ้งมวลความน่าจะเป็นไปไม่ถึง 1%
(ตัวเลข coverage จริงพิมพ์ไว้แล้วในเซลล์ก่อนหน้า และจะวัดซ้ำบนข้อมูลเทรนจริงอีกครั้ง)


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 6.1 -- โหลด 3,000 prompt แล้วจัดรูปสำหรับนักเรียน base (ไม่มี chat template)
# -----------------------------------------------------------------------------
from datasets import load_dataset

N_PROMPTS = 3000        # คลังตั้งต้นที่สุ่มมาจากชุดข้อมูล
N_TRAIN = 384           # ใช้เทรนจริงต่อ regime (= 24 step ที่ effective batch 16)
N_PROBE = 8             # ชุดสาธิตสำหรับ samples.json -- แยกขาดจากชุดเทรน
MAX_LEN = 512           # ความยาวรวมสูงสุดต่อประโยค (token)
MAX_PROMPT_LEN = 256    # โควตาฝั่งคำถาม

PROMPT_TMPL = "คำถาม: {q}\nคำตอบ: "


def pick_column(columns, candidates):
    lower = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand in lower:
            return lower[cand]
    return None


raw = load_dataset("airesearch/wangchanx-seed-free-synthetic-instruct-thai-120k", split="train")
print(f"wangchanx columns: {raw.column_names}  n={len(raw)}")

q_col = pick_column(raw.column_names, ["instruction", "question", "input", "prompt"])
a_col = pick_column(raw.column_names, ["output", "response", "answer", "completion"])
assert None not in (q_col, a_col), f"schema ไม่ตรงที่คาด: {raw.column_names}"

raw = raw.shuffle(seed=SEED).select(range(min(N_PROMPTS, len(raw))))
pairs = []
for row in raw:
    q, a = row[q_col], row[a_col]
    if not (isinstance(q, str) and isinstance(a, str)):
        continue
    q, a = q.strip(), a.strip()
    if 10 <= len(q) <= 300 and 40 <= len(a) <= 700:
        pairs.append((q, a))
    if len(pairs) >= N_TRAIN + N_PROBE:
        break

assert len(pairs) >= N_TRAIN + N_PROBE, "prompt ผ่านการคัดกรองไม่พอ -- ลด N_TRAIN ลง"
train_pairs = pairs[:N_TRAIN]
probe_pairs = pairs[N_TRAIN:N_TRAIN + N_PROBE]
probe_questions = [q for q, _ in probe_pairs]


def encode_example(q, a):
    """คืน dict {input_ids, labels} โดย mask ฝั่งคำถามเป็น -100 (บทเรียนตอนที่ 2)"""
    p_ids = tokenizer(PROMPT_TMPL.format(q=q), add_special_tokens=False).input_ids
    p_ids = p_ids[:MAX_PROMPT_LEN]
    a_ids = tokenizer(a, add_special_tokens=False).input_ids + [tokenizer.eos_token_id]
    a_ids = a_ids[: MAX_LEN - len(p_ids)]
    if len(a_ids) < 8:
        return None
    return {"input_ids": p_ids + a_ids, "labels": [-100] * len(p_ids) + a_ids}


gold_train = [ex for ex in (encode_example(q, a) for q, a in train_pairs) if ex]
lens = [len(ex["input_ids"]) for ex in gold_train]
mean_ans_th = sum(th_ratio(a) for _, a in train_pairs) / len(train_pairs)

print(f"\nคัดกรองแล้ว: เทรน {len(gold_train)} ข้อ | probe {len(probe_questions)} ข้อ")
print(f"ความยาวประโยคเทรน: เฉลี่ย {sum(lens) / len(lens):.0f} token | สูงสุด {max(lens)}")
print(f"th_ratio เฉลี่ยของเฉลยจริง: {mean_ans_th:.3f}")
print()
print("ตัวอย่างข้อมูลหนึ่งข้อ:")
print("  คำถาม:", train_pairs[0][0][:100].replace(chr(10), " "))
print("  เฉลย  :", train_pairs[0][1][:100].replace(chr(10), " "))
n_step = len(gold_train) // 16
print(f"\nที่ effective batch = 16 จะได้ {n_step} optimizer step ต่อ regime")


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 6.2 -- คณิตศาสตร์ของหน่วยความจำ: ทำไม logits เต็ม vocab เก็บไม่ได้จริง
# เซลล์นี้เป็นเลขคณิตล้วน ๆ จากค่า config จริง (VOCAB_SIZE อ่านมาจากโมเดล ไม่ใช่จำมา)
# -----------------------------------------------------------------------------
TOP_K = 64
B_DEMO = 4                      # batch สมมติหนึ่งก้อน
BYTES_FP16 = 2
BYTES_INT32 = 4

full_batch = B_DEMO * MAX_LEN * VOCAB_SIZE * BYTES_FP16
topk_batch = B_DEMO * MAX_LEN * TOP_K * (BYTES_FP16 + BYTES_INT32)

print(f"logits ครูเต็ม vocab หนึ่ง batch  = {B_DEMO} x {MAX_LEN} x {VOCAB_SIZE:,} x {BYTES_FP16} B")
print(f"                                = {full_batch:,} bytes = {full_batch / 1e6:.1f} MB")
print(f"top-{TOP_K} (ค่า fp16 + ดัชนี int32)  = {B_DEMO} x {MAX_LEN} x {TOP_K} x {BYTES_FP16 + BYTES_INT32} B")
print(f"                                = {topk_batch:,} bytes = {topk_batch / 1e6:.2f} MB")
print(f"เล็กลง                           = {full_batch / topk_batch:.0f} เท่า")

full_offline = N_PROMPTS * MAX_LEN * VOCAB_SIZE * BYTES_FP16
topk_offline = N_PROMPTS * MAX_LEN * TOP_K * (BYTES_FP16 + BYTES_INT32)
print(f"\nถ้าเก็บ offline ทั้ง {N_PROMPTS:,} ตัวอย่าง:")
print(f"  เต็ม vocab : {full_offline / 1e9:,.1f} GB   <- ไม่มีดิสก์ Colab ไหนรับไหว")
print(f"  top-{TOP_K}     : {topk_offline / 1e9:.2f} GB")

# รายละเอียดเล็ก ๆ ที่สอนอะไรได้เยอะ: ดัชนีต้องเป็น int32 เพราะ vocab เกินเพดาน uint16
assert VOCAB_SIZE > 65535, "ถ้า vocab เล็กกว่านี้ ใช้ uint16 เก็บดัชนีได้ถูกกว่าครึ่งหนึ่ง"
print(f"\nดัชนี vocab สูงสุด {VOCAB_SIZE - 1:,} > 65,535 (เพดาน uint16)")
print("-> ดัชนีต้องใช้ int32 (4 B) ซึ่งใหญ่กว่าตัวค่า logits fp16 (2 B) เสียอีก")
print("การประหยัดจึงมาจากการตัดจำนวนช่อง ไม่ใช่จากการบีบชนิดข้อมูล")


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 6.3 -- วัตถุดิบชิ้นที่ 2: คำตอบของครู (สำหรับ SeqKD) -- generate ครั้งเดียวเก็บไว้
# -----------------------------------------------------------------------------
GEN_BATCH = 16
GEN_MAX_NEW = 96


def teacher_prompt(q):
    """ครูเป็นโมเดล instruct จึงคุยผ่าน chat template (ปิด thinking mode ตามสัญญาซีรีส์)"""
    return tok_teacher.apply_chat_template(
        [{"role": "user", "content": q}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


gen_questions = [q for q, _ in train_pairs] + probe_questions
tok_teacher.padding_side = "left"       # decoder-only ต้อง pad ซ้ายตอน generate
teacher_answers = []
t0 = time.time()
with torch.no_grad():
    for i in range(0, len(gen_questions), GEN_BATCH):
        chunk = gen_questions[i:i + GEN_BATCH]
        enc = tok_teacher([teacher_prompt(q) for q in chunk], return_tensors="pt",
                          padding=True, truncation=True, max_length=MAX_PROMPT_LEN).to(DEV)
        torch.manual_seed(SEED)
        out = teacher.generate(
            **enc,
            max_new_tokens=GEN_MAX_NEW,
            do_sample=False,            # greedy -- ทำซ้ำได้
            pad_token_id=tok_teacher.pad_token_id,
        )
        new_tokens = out[:, enc["input_ids"].shape[1]:]
        texts = tok_teacher.batch_decode(new_tokens, skip_special_tokens=True)
        teacher_answers.extend(t.split("</think>")[-1].strip() for t in texts)
        if (i // GEN_BATCH) % 4 == 0:
            print(f"  generate {i + len(chunk)}/{len(gen_questions)} ({time.time() - t0:.0f}s)")
tok_teacher.padding_side = "right"
print(f"ครู generate {len(teacher_answers)} คำตอบ ใน {time.time() - t0:.0f} วินาที")

train_teacher_answers = teacher_answers[:len(train_pairs)]
teacher_probe_answers = dict(zip(probe_questions, teacher_answers[len(train_pairs):]))

seqkd_train = [
    ex for ex in (encode_example(q, a)
                  for (q, _), a in zip(train_pairs, train_teacher_answers) if a)
    if ex
]
mean_teacher_th = sum(th_ratio(a) for a in train_teacher_answers) / len(train_teacher_answers)

print(f"\nประโยคเทรน SeqKD ที่ใช้ได้: {len(seqkd_train)} ข้อ")
print(f"th_ratio เฉลี่ยของคำตอบครู: {mean_teacher_th:.3f} (เทียบเฉลยจริง {mean_ans_th:.3f})")
print("ถ้าครูหลุดภาษาอังกฤษบ่อย นักเรียน SeqKD จะรับนิสัยนั้นมาเต็ม ๆ -- จดตัวเลขนี้ไว้")
print()
print("ตัวอย่างคำตอบครู:", train_teacher_answers[0][:150].replace(chr(10), " "))


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 6.4 -- วัตถุดิบชิ้นที่ 3: top-64 logits ของครู (offline, ครั้งเดียวจบ)
# จุดที่สำคัญที่สุดของเซลล์นี้คือ torch.no_grad() -- ถ้าลืม autograd จะเก็บ
# activation ของครูทั้งตัวรอ backward ที่ไม่มีวันมาถึง แล้ว OOM โดยชี้ไปบรรทัดอื่น
# -----------------------------------------------------------------------------
T_COVERAGE = 2.0
PRECOMP_BATCH = 4

teacher_topk_store = []      # ต่อประโยค: (val [L,64] fp16, idx [L,64] int32) บน CPU
coverage_samples = []

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
with torch.no_grad():
    for i in range(0, len(gold_train), PRECOMP_BATCH):
        chunk = gold_train[i:i + PRECOMP_BATCH]
        width = max(len(ex["input_ids"]) for ex in chunk)
        ids = torch.full((len(chunk), width), tokenizer.pad_token_id, dtype=torch.long)
        att = torch.zeros((len(chunk), width), dtype=torch.long)
        for j, ex in enumerate(chunk):
            n = len(ex["input_ids"])
            ids[j, :n] = torch.tensor(ex["input_ids"], dtype=torch.long)
            att[j, :n] = 1
        z = teacher(input_ids=ids.to(DEV), attention_mask=att.to(DEV)).logits  # [B, L, V]
        val, idx = z.topk(TOP_K, dim=-1)
        if i == 0:
            # วัด coverage จริงบนข้อมูลเทรน (เฉพาะ batch แรก พอเป็นตัวแทน)
            p_full = torch.softmax(z.float() / T_COVERAGE, dim=-1)
            cov = p_full.gather(-1, idx).sum(-1)                    # [B, L]
            m = att.to(DEV).bool()
            coverage_samples.append(float(cov[m].mean()))
        for j, ex in enumerate(chunk):
            n = len(ex["input_ids"])
            teacher_topk_store.append(
                (val[j, :n].half().cpu(), idx[j, :n].to(torch.int32).cpu())
            )
        del z, val, idx

peak = torch.cuda.max_memory_allocated() / (1024 ** 3)
store_mb = sum(v.numel() * 2 + x.numel() * 4 for v, x in teacher_topk_store) / 1e6
print(f"forward ครู {len(gold_train)} ประโยค ใน {time.time() - t0:.0f} วินาที")
print(f"VRAM peak ระหว่าง precompute: {peak:.2f} GB "
      f"(สังเกตก้อน logits ชั่วคราวตามเลขคณิตหัวข้อ 6.2)")
print(f"ขนาด store บน CPU: {store_mb:.0f} MB (เทียบเต็ม vocab จะเป็นระดับหลายสิบ GB)")
print(f"coverage ของ top-{TOP_K} ที่ T={T_COVERAGE:g} บนข้อมูลเทรนจริง: "
      f"{100 * coverage_samples[0]:.2f}%")
assert teacher_topk_store[0][1].dtype == torch.int32   # ดัชนีต้อง int32 (หัวข้อ 6.2)


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 6.5 -- tokens.json: ครูกับนักเรียนมองตำแหน่งเดียวกัน ต่างกันแค่ไหน
# (วิดเจ็ต TokenProbInspector บนบล็อกอ่านไฟล์นี้: "ก่อน" = นักเรียน, "หลัง" = ครู)
# เสร็จแล้วไล่ครูลงจากการ์ด -- หน้าที่บนการ์ดของครูจบแล้ว
# -----------------------------------------------------------------------------
THAI_SENT = "เมืองหลวงของประเทศไทยคือกรุงเทพมหานคร และภาษาราชการคือภาษาไทย"
sent_ids = tokenizer(THAI_SENT, add_special_tokens=False).input_ids
ids_t = torch.tensor([sent_ids], device=DEV)

with torch.no_grad():
    logp_teacher = torch.log_softmax(teacher(input_ids=ids_t).logits[0].float(), dim=-1)
    logp_student = torch.log_softmax(student(input_ids=ids_t).logits[0].float(), dim=-1)


def top5_at(logp, pos):
    val, idx = logp[pos].topk(5)
    return [{"token": tokenizer.decode([int(i)]), "logprob": round(float(v), 4)}
            for v, i in zip(val, idx)]


tokens, lp_before, lp_after, t5_before, t5_after = [], [], [], [], []
for pos in range(len(sent_ids) - 1):          # ตำแหน่ง pos ทำนาย token ที่ pos+1
    nxt = sent_ids[pos + 1]
    tokens.append(tokenizer.decode([nxt]))
    lp_before.append(round(float(logp_student[pos, nxt]), 4))
    lp_after.append(round(float(logp_teacher[pos, nxt]), 4))
    t5_before.append(top5_at(logp_student, pos))
    t5_after.append(top5_at(logp_teacher, pos))

with open("tokens.json", "w", encoding="utf-8") as f:
    json.dump({
        "prompt": THAI_SENT,
        "tokens": tokens,
        "logprobs_before": lp_before,   # นักเรียน (ก่อนกลั่น)
        "logprobs_after": lp_after,     # ครู -- ช่องว่างระหว่างสองมุมมองคือสิ่งที่ KD พยายามปิด
        "top5_before": t5_before,
        "top5_after": t5_after,
    }, f, ensure_ascii=False, indent=1)

print(f"เขียน tokens.json แล้ว ({len(tokens)} ตำแหน่ง token ภาษาไทย)")
print(f"{'token':14s} {'logp นักเรียน':>14s} {'logp ครู':>10s}   top-1 ของครู")
for k in range(min(6, len(tokens))):
    print(f"{tokens[k]!r:14s} {lp_before[k]:14.2f} {lp_after[k]:10.2f}   "
          f"{t5_after[k][0]['token']!r}")

# --- ครูจบหน้าที่บนการ์ดแล้ว: logits อยู่ใน store, คำตอบอยู่ใน list ---
del teacher
n_collected = gc.collect()
torch.cuda.empty_cache()
VRAM_AFTER_FREE = torch.cuda.memory_allocated() / (1024 ** 3)
print(f"\nคืน VRAM ของครูแล้ว (gc {n_collected} objects): "
      f"{VRAM_BOTH_GB:.2f} GB -> {VRAM_AFTER_FREE:.2f} GB")
print("จากนี้ไปการเทรนทั้งหมดใช้ logits จาก store -- นี่คือความหมายของคำว่า offline KD")


---

## 7. โค้ดหลัก (Main code)

### 7.1 เขียน KD loss เองด้วยมือ — สมการ 3.1 ทีละบรรทัด

สี่จุดที่ต้องทำให้ถูก ไม่งั้นสมการ 3.1 จะไม่ใช่สมการ 3.1 อีกต่อไป:

1. **หาร $T$ ทั้งสองฝั่ง** — ถ้าหารแค่ฝั่งครู นักเรียนจะถูกบังคับให้เลียนแบบ distribution
   แบน ๆ ด้วย logits คม ๆ ของตัวเอง ผลคือมันเรียนรู้ที่จะ "แบนจริง ๆ" แล้วตอนใช้งาน
   (ซึ่งไม่มี $T$) คำตอบจะจืดและกระจายผิดปกติ
2. **คูณ $T^2$ คืน** — ไม่งั้นการกวาดหา $T$ ที่ดีที่สุดคือการกวาด learning rate
   ไปพร้อมกันโดยไม่ตั้งใจ (สมการ 3.2 และการวัดจริงในเซลล์ T-sweep)
3. **mask padding และฝั่งคำถามออกจาก KL** — ตำแหน่ง padding ก็มี distribution ของครู
   เหมือนกัน และมันคือขยะ ถ้าไม่ mask ค่าเฉลี่ย KL จะถูกเจือจางไม่เท่ากันตามความยาว
   ประโยคใน batch — loss จะสั่นแบบหาสาเหตุไม่เจอ
4. **logits ครูต้องไม่มี gradient graph** — เส้นทาง offline ของเราปลอดภัยโดยโครงสร้าง
   (มาจากดิสก์/CPU store ไม่มี graph ให้เก็บ) แต่เราจะ assert ซ้ำให้เห็น

### 7.2 พิสูจน์สองชั้น โดยไม่ต้องใช้ GPU และไม่ต้องใช้โมเดลจริง

**ชั้นที่ 1 — float64 อิสระ:** เขียนสมการ 3.1 ซ้ำอีกครั้งด้วย `math.exp` + list ล้วน ๆ
คนละสไตล์ คนละบรรทัด แล้ว assert ว่าตรงกับเวอร์ชัน torch ภายใน `1e-6`
บน tensor สุ่มที่มีทั้ง prompt ที่ถูก mask และ padding ปลอม ๆ
ถ้าการ shift ตำแหน่ง, การ mask หรือการ renormalize บน top-K ผิดแม้จุดเดียว
ตัวเลขสองทางจะแยกจากกันทันที

**ชั้นที่ 2 — T-sweep:** วัด `grad.norm()` ของพจน์ soft จริง ๆ ที่ $T = 1 \ldots 16$
ต้องเห็น gradient ไหลลงตาม $1/T^2$ เมื่อไม่คูณ $T^2$ (slope log-log ≈ −2)
และแทบแบนเมื่อคูณคืน — นี่คือสมการ 3.2 ในรูปแบบที่ autograd เป็นพยานให้

สองเซลล์ถัดไปรันได้แม้บนเครื่องไม่มี GPU — มันคือหัวใจที่ตรวจสอบได้ของทั้งตอน


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 7.1-7.2 -- KD loss เขียนเอง + การพิสูจน์กับ float64 ที่เขียนแยกอิสระ
# เซลล์นี้ไม่ใช้ GPU และไม่ใช้โมเดลจริง
# -----------------------------------------------------------------------------
T_KD = 2.0     # temperature (สมการ 3.1)
ALPHA = 0.9    # น้ำหนักพจน์ soft: ฟังครูเป็นหลัก มีเฉลยจริง 10% กันหลุด


def _up(x):
    """fp16 -> fp32 เพื่อความเสถียรบน GPU; fp32/fp64 ปล่อยผ่านตามเดิม
    (สำคัญต่อการพิสูจน์: ถ้าบังคับ fp32 เสมอ การเทียบกับ float64 จะโดนการปัดเศษ
    ของ fp32 บังจนไม่มีวันเห็นว่าสูตร "ตรงจริง" ระดับ 1e-12)"""
    return x.float() if x.dtype == torch.float16 else x


def kd_loss(student_logits, teacher_val, teacher_idx, labels, T=T_KD, alpha=ALPHA):
    """สมการ 3.1 บนแกน top-K ของครู -- ทั้ง KD อยู่ในฟังก์ชันนี้แล้ว ไม่มีอะไรซ่อนอีก

    student_logits : [B, L, V]  logits นักเรียน (มี gradient)
    teacher_val    : [B, L, K]  ค่า top-K logits ของครู (จาก store -- ไม่มี graph)
    teacher_idx    : [B, L, K]  ดัชนี vocab ของ top-K
    labels         : [B, L]     เฉลยจริง; ฝั่งคำถามและ padding เป็น -100
    """
    z_s = _up(student_logits)[:, :-1]            # ตำแหน่ง t ทำนาย token ที่ t+1
    t_val = _up(teacher_val)[:, :-1]
    t_idx = teacher_idx[:, :-1]
    tgt = labels[:, 1:]
    mask = tgt.ne(-100)                          # จุดที่ 3: กันคำถาม + padding ปน loss

    # -- พจน์ hard: cross-entropy กับเฉลยจริง เหมือน SFT ทุกประการ --
    ce = F.cross_entropy(z_s.flatten(0, 1), tgt.flatten(), ignore_index=-100)

    # -- พจน์ soft: KL(ครู || นักเรียน) บนแกน top-K -- หาร T "ทั้งสองฝั่ง" (จุดที่ 1) --
    p_t = F.softmax(t_val / T, dim=-1)                        # renormalize บน K มิติ
    z_k = z_s.gather(-1, t_idx.long())                        # logits นักเรียนช่องเดียวกัน
    log_q = F.log_softmax(z_k / T, dim=-1)
    kl = (p_t * (p_t.clamp_min(1e-9).log() - log_q)).sum(-1)  # KL ต่อตำแหน่ง
    kl = (kl * mask).sum() / mask.sum().clamp_min(1)          # เฉลี่ยเฉพาะ token คำตอบ

    return (1 - alpha) * ce + alpha * (T ** 2) * kl           # จุดที่ 2: T^2 จากสมการ 3.2


def kd_loss_ref_float64(student_logits, teacher_val, teacher_idx, labels, T=T_KD, alpha=ALPHA):
    """สมการ 3.1 อีกครั้ง ด้วย float64 + list + math.exp ล้วน ๆ (ไม่แตะ F.* เลย)
    เขียนแยกอิสระเพื่อให้สองทางจับ bug ของกันและกัน"""
    B, L, V = student_logits.shape
    K = teacher_val.shape[-1]
    ce_sum = kl_sum = 0.0
    ce_n = kl_n = 0
    for b in range(B):
        for t in range(L - 1):
            tgt = int(labels[b, t + 1])
            if tgt == -100:
                continue
            row = [float(student_logits[b, t, v]) for v in range(V)]
            m = max(row)
            log_z = m + math.log(sum(math.exp(x - m) for x in row))
            ce_sum += log_z - row[tgt]
            ce_n += 1
            tv = [float(teacher_val[b, t, k]) / T for k in range(K)]
            mt = max(tv)
            zt = sum(math.exp(x - mt) for x in tv)
            p = [math.exp(x - mt) / zt for x in tv]
            sv = [float(student_logits[b, t, int(teacher_idx[b, t, k])]) / T for k in range(K)]
            ms = max(sv)
            log_zs = ms + math.log(sum(math.exp(x - ms) for x in sv))
            kl_sum += sum(pi * (math.log(max(pi, 1e-9)) - (x - log_zs))
                          for pi, x in zip(p, sv))
            kl_n += 1
    ce = ce_sum / max(ce_n, 1)
    kl = kl_sum / max(kl_n, 1)
    return (1 - alpha) * ce + alpha * (T ** 2) * kl


# --- การพิสูจน์ชั้นที่ 1: torch vs float64 อิสระ บน tensor สุ่มขนาดเล็ก ---
torch.manual_seed(SEED)
_B, _L, _V, _K = 2, 9, 23, 5
_z_s = torch.randn(_B, _L, _V, dtype=torch.float64) * 2.0
_t_val, _t_idx = (torch.randn(_B, _L, _V, dtype=torch.float64) * 2.0).topk(_K, dim=-1)
_labels = torch.randint(0, _V, (_B, _L))
_labels[:, :3] = -100        # จำลองฝั่งคำถามที่ถูก mask
_labels[1, -2:] = -100       # จำลอง padding ท้ายประโยค

loss_torch = kd_loss(_z_s, _t_val, _t_idx, _labels)
loss_ref = kd_loss_ref_float64(_z_s, _t_val, _t_idx, _labels)
diff = abs(float(loss_torch) - loss_ref)

print(f"kd_loss (torch)   = {float(loss_torch):.12f}")
print(f"kd_loss (float64) = {loss_ref:.12f}")
print(f"|ผลต่าง|           = {diff:.3e}")
assert diff < 1e-6, "สองทางอิสระให้ค่าไม่ตรงกัน -- สูตรผิดที่ใดที่หนึ่ง"
print("=> ผ่าน: implementation อิสระสองทางตรงกันระดับ float64")

# mask ต้องเปลี่ยนตัวเลขจริง ไม่ใช่โค้ดประดับ
_labels_nomask = _labels.clone()
_labels_nomask[_labels_nomask == -100] = 0
loss_nomask = kd_loss(_z_s, _t_val, _t_idx, _labels_nomask)
assert abs(float(loss_nomask) - float(loss_torch)) > 1e-6
print(f"ถ้าไม่ mask: loss = {float(loss_nomask):.6f} (เปลี่ยนจริง -> จุดที่ 3 ไม่ใช่เรื่องเล็ก)")

# จุดที่ 4: logits ครูไม่มี graph และ gradient ไหลเข้าฝั่งนักเรียนเท่านั้น
assert not _t_val.requires_grad
_z_grad = _z_s.clone().requires_grad_(True)
kd_loss(_z_grad, _t_val, _t_idx, _labels).backward()
assert _z_grad.grad is not None and torch.isfinite(_z_grad.grad).all()
print("gradient ไหลเข้าฝั่งนักเรียนเท่านั้น และ finite ทุกตัว -> ผ่าน")


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 7.2 (ต่อ) -- การพิสูจน์ชั้นที่ 2: T-sweep วัด gradient จริง
# เซลล์นี้ไม่ใช้ GPU และไม่ใช้โมเดลจริง
# -----------------------------------------------------------------------------
torch.manual_seed(SEED)
_Bs, _Ls, _Vs = 4, 8, 64
z_teacher_demo = torch.randn(_Bs, _Ls, _Vs, dtype=torch.float64) * 3.0
z_student_demo = torch.randn(_Bs, _Ls, _Vs, dtype=torch.float64) * 3.0

TS = [1.0, 2.0, 4.0, 8.0, 16.0]
grad_raw, grad_fixed = [], []
for T in TS:
    z_s2 = z_student_demo.clone().requires_grad_(True)
    p = F.softmax(z_teacher_demo / T, dim=-1)
    log_q = F.log_softmax(z_s2 / T, dim=-1)
    kl = (p * (p.clamp_min(1e-9).log() - log_q)).sum(-1).mean()   # soft loss "ลืม" T^2
    kl.backward()
    g = float(z_s2.grad.norm())
    grad_raw.append(g)
    grad_fixed.append(T * T * g)      # คูณ T^2 คืน (T เป็นค่าคงที่ จึงเท่ากับ grad ของ T^2*KL)

print(f"{'T':>5s} {'grad ไม่คูณ T^2':>18s} {'grad คูณ T^2':>15s}")
for T, gr, gf in zip(TS, grad_raw, grad_fixed):
    print(f"{T:5.1f} {gr:18.6f} {gf:15.6f}")

ratio_raw = grad_raw[0] / grad_raw[-1]
ratio_fixed = max(grad_fixed) / min(grad_fixed)
slope = (math.log(grad_raw[-1]) - math.log(grad_raw[2])) / (math.log(TS[-1]) - math.log(TS[2]))
print(f"\ngrad(T=1)/grad(T=16) ไม่คูณ T^2 = {ratio_raw:.1f}  (สเกล ~T^2 = 256)")
print(f"max/min ของฝั่งคูณ T^2 แล้ว     = {ratio_fixed:.2f}  (อุดมคติ ~1)")
print(f"slope log-log ช่วง T=4..16      = {slope:.3f}  (ทฤษฎีบอก -2)")
assert ratio_raw > 50, "gradient ไม่ได้ไหลลงตาม 1/T^2 -- สมการ 3.2 มีปัญหา"
assert ratio_fixed < 3.0, "คูณ T^2 แล้วยังไม่นิ่ง -- มีอะไรผิดในสูตร"
assert -2.5 < slope < -1.5, "slope ไม่เข้าใกล้ -2 ตามทฤษฎี"
print("=> ผ่านทั้งสามข้อ: ตัวคูณ T^2 ไม่ใช่เครื่องราง มันคือการชดเชย gradient ที่วัดได้จริง")

use_thai_font()
fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.plot(TS, grad_raw, "o-", color="#dc2626", lw=2, label="ไม่คูณ T² (ไหลลงตาม 1/T²)")
ax.plot(TS, grad_fixed, "s-", color="#16a34a", lw=2, label="คูณ T² คืน (นิ่ง)")
ax.plot(TS, [grad_raw[0] / (T ** 2) for T in TS], ":", color="#64748b", lw=1.5,
        label="เส้นอ้างอิง 1/T²")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xticks(TS)
ax.set_xticklabels([f"{t:g}" for t in TS])
ax.set_xlabel("temperature T")
ax.set_ylabel("‖gradient ของ soft loss‖ (log)")
ax.set_title("ลืม T² = แอบหาร learning rate ของ soft loss ด้วย T²", fontsize=11)
ax.legend()
ax.grid(alpha=0.3, which="both")
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig("fig_t2_gradient.png", dpi=150, bbox_inches="tight")
plt.show()
print("บันทึกรูปแล้ว: fig_t2_gradient.png")


### 7.3 LoRA สอง adapter บนนักเรียนตัวเดียว + Trainer ที่ป้อน logits ครูจาก store

สอง regime ต้องเทียบกันได้ จึงต้องออกตัวจากนักเรียน **ตัวเดียวกันเป๊ะ** เราใช้ความสามารถ
multi-adapter ของ peft: adapter `"seqkd"` กับ `"kd"` เกาะอยู่บนน้ำหนักฐานก้อนเดียว
สลับกันด้วย `set_adapter()` — เทรนอิสระจากกัน ไม่ปนเปื้อนกัน และ "ก่อนกลั่น"
ก็คือการปิด adapter ทั้งคู่ (`disable_adapter()`) — ไม่ต้องโหลดโมเดลใหม่แม้แต่ครั้งเดียว

กับดักเดิมจากตอนที่ 2/4 ยังอยู่ครบ:

- **cast เฉพาะพารามิเตอร์ adapter เป็น fp32** ก่อนเทรน ไม่งั้น `GradScaler` โยน
  `ValueError: Attempting to unscale FP16 gradients.` และต้อง cast ซ้ำทุกครั้งที่สลับ adapter
  เพราะ adapter ที่เพิ่งถูกปลุกอาจยังเป็น fp16 อยู่
- **`remove_unused_columns=False`** — โดยปกติ `Trainer` จะทิ้งคอลัมน์ที่ signature
  ของโมเดลไม่รู้จัก ซึ่งรวมถึง `teacher_val`/`teacher_idx` ของเราด้วย
  อาการคือ `KeyError` ตั้งแต่ step แรก (โชคดีที่พังดัง ไม่ได้พังเงียบ)
- **lr = 1e-4** — สูงได้เพราะเทรนแค่ adapter (น้ำหนักฐานแช่แข็ง)
  ต่ำกว่า SFT LoRA ตอนที่ 2 (2e-4) เล็กน้อย เพราะพจน์ KL ให้ gradient หนาแน่นกว่า CE
  (ทุกตำแหน่งมี 64 เป้าหมาย ไม่ใช่เป้าเดียว)


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 7.3 -- LoRA สอง adapter, dataset/collator และ KDTrainer
# -----------------------------------------------------------------------------
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments

BATCH_SIZE = 2
GRAD_ACCUM = 8            # effective batch = 16
LR = 1e-4


def make_lora_cfg():
    return LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
    )


student = get_peft_model(student, make_lora_cfg(), adapter_name="seqkd")
student.add_adapter("kd", make_lora_cfg())
student.add_adapter("sft", make_lora_cfg())   # แถวควบคุม: เฉลยจริง + CE ล้วน ไม่มี distribution ครู


def cast_trainable_to_fp32(m):
    n_cast = 0
    for _, p in m.named_parameters():
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.float()
            n_cast += 1
    return n_cast


def activate_adapter(name):
    """สลับ adapter + ตรึงให้เทรนได้เฉพาะ adapter นั้น + cast เป็น fp32 (กับดัก fp16)"""
    student.set_adapter(name)
    for n, p in student.named_parameters():
        p.requires_grad = ("lora_" in n and f".{name}." in n)
    n_cast = cast_trainable_to_fp32(student)
    n_train = sum(p.numel() for p in student.parameters() if p.requires_grad)
    print(f"adapter '{name}': เทรนได้ {n_train / 1e6:.2f}M พารามิเตอร์ "
          f"(cast fp32 ไป {n_cast} tensors)")


class KDDataset(torch.utils.data.Dataset):
    """ห่อ list ของตัวอย่าง; ใส่ logits ครูให้เฉพาะ regime ที่ต้องใช้"""

    def __init__(self, examples, teacher_store=None):
        self.examples = examples
        self.teacher_store = teacher_store

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        item = dict(self.examples[i])
        if self.teacher_store is not None:
            val, idx = self.teacher_store[i]
            item["teacher_val"], item["teacher_idx"] = val, idx
        return item


def collate(features):
    """right-pad ทุกอย่างให้กว้างเท่ากัน -- labels เป็น -100 บน padding เสมอ"""
    width = max(len(f["input_ids"]) for f in features)
    pad = tokenizer.pad_token_id
    ids = torch.full((len(features), width), pad, dtype=torch.long)
    att = torch.zeros((len(features), width), dtype=torch.long)
    lab = torch.full((len(features), width), -100, dtype=torch.long)
    for i, f in enumerate(features):
        n = len(f["input_ids"])
        ids[i, :n] = torch.tensor(f["input_ids"], dtype=torch.long)
        att[i, :n] = 1
        lab[i, :n] = torch.tensor(f["labels"], dtype=torch.long)
    batch = {"input_ids": ids, "attention_mask": att, "labels": lab}
    if "teacher_val" in features[0]:
        K = features[0]["teacher_val"].shape[-1]
        tval = torch.zeros((len(features), width, K), dtype=torch.float16)
        tidx = torch.zeros((len(features), width, K), dtype=torch.long)
        for i, f in enumerate(features):
            n = f["teacher_val"].shape[0]
            tval[i, :n] = f["teacher_val"]
            tidx[i, :n] = f["teacher_idx"].long()
        batch["teacher_val"], batch["teacher_idx"] = tval, tidx
    return batch


class KDTrainer(Trainer):
    """Trainer ปกติทุกอย่าง ยกเว้น loss ที่เปลี่ยนเป็น kd_loss ของเรา"""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        t_val = inputs.pop("teacher_val")    # มาจาก store ไม่ใช่จากครูบนการ์ด
        t_idx = inputs.pop("teacher_idx")
        out = model(input_ids=inputs["input_ids"],
                    attention_mask=inputs["attention_mask"])
        loss = kd_loss(out.logits, t_val, t_idx, inputs["labels"], T=T_KD, alpha=ALPHA)
        return (loss, out) if return_outputs else loss


def make_args(tag):
    return TrainingArguments(
        output_dir=f"out-{tag}",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=1,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        max_grad_norm=1.0,
        fp16=USE_FP16,                  # บน T4 ค่านี้คือ True (มาจาก Cell 0)
        bf16=SUPPORTS_BF16,             # บน T4 ค่านี้คือ False
        logging_steps=5,
        save_strategy="no",
        report_to="none",
        seed=SEED,
        remove_unused_columns=False,    # ห้ามลืม -- ไม่งั้น teacher_val/teacher_idx หาย
    )


def run_regime(tag, dataset, trainer_cls):
    activate_adapter(tag)
    student.config.use_cache = False
    student.train()
    trainer = trainer_cls(model=student, args=make_args(tag),
                          train_dataset=dataset, data_collator=collate)
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    trainer.train()
    student.config.use_cache = True
    student.eval()
    run = {
        "tag": tag,
        "train_time_s": time.time() - t0,
        "vram_peak_gb": torch.cuda.max_memory_allocated() / (1024 ** 3),
        "log_history": [dict(h) for h in trainer.state.log_history],
    }
    print(f"[{tag}] เทรน {run['train_time_s'] / 60:.1f} นาที | "
          f"VRAM peak {run['vram_peak_gb']:.2f} GB")
    return run


seqkd_ds = KDDataset(seqkd_train)
kd_ds = KDDataset(gold_train, teacher_store=teacher_topk_store)
print(f"\nพร้อมเทรน: SeqKD {len(seqkd_ds)} ข้อ | logit KD {len(kd_ds)} ข้อ "
      f"({len(kd_ds) // (BATCH_SIZE * GRAD_ACCUM)} step ต่อ regime)")


---

## 8. ผลลัพธ์ (Results)

รันสอง regime ด้วย adapter คนละตัวบนนักเรียนตัวเดียวกัน จำนวน step เท่ากัน
hyperparameter เดียวกันทุกตัว **ต่างกันแค่สัญญาณที่ใช้สอน**:

- **SeqKD** — cross-entropy บนคำตอบที่ครู generate (ข้อความจากครู)
- **Logit KD** — สมการ 3.1 เต็ม ๆ: CE บนเฉลยจริง + KL บน top-64 ของครู (distribution จากครู)

คำเตือนตอนอ่าน log: **อย่าเทียบตัวเลข loss ระหว่างสอง regime ตรง ๆ**
มันคือ loss คนละนิยาม — ฝั่ง KD มีพจน์ KL คูณ $T^2$ บวกอยู่ ตัวเลขจึงสูงกว่าโดยธรรมชาติ
สิ่งที่เทียบได้คือแนวโน้ม (ลดหรือไม่ลด) และผลวัด KobEval-TH ในหัวข้อ 9 เท่านั้น


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 8.1 -- regime ที่ 1: SeqKD (SFT บนคำตอบครู -- Trainer ปกติ ไม่มีอะไรใหม่)
# -----------------------------------------------------------------------------
# regime ควบคุม: SFT บนเฉลยจริง (CE ล้วน) -- เทรนบน gold_train เดียวกับ logit KD
# ต่างจาก logit KD ข้อเดียว: ไม่มี distribution ครู -> ส่วนต่าง = มูลค่า dark knowledge ล้วน
SFT_RUN = run_regime("sft", KDDataset(gold_train), Trainer)

SEQKD_RUN = run_regime("seqkd", seqkd_ds, Trainer)


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 8.2 -- regime ที่ 2: offline logit KD (KDTrainer + kd_loss ที่พิสูจน์แล้ว)
# -----------------------------------------------------------------------------
KD_RUN = run_regime("kd", kd_ds, KDTrainer)

use_thai_font()
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for run, color, label in [(SEQKD_RUN, "#2563eb", "SeqKD (CE บนคำตอบครู)"),
                          (KD_RUN, "#dc2626", "logit KD (CE + T²·KL)")]:
    pts = [(h["step"], h["loss"]) for h in run["log_history"] if "loss" in h]
    if pts:
        ax.plot([p[0] for p in pts], [p[1] for p in pts], "-o", ms=3,
                color=color, label=label)
ax.set_xlabel("optimizer step")
ax.set_ylabel("training loss")
ax.set_title("คนละนิยาม loss -- เทียบแนวโน้มได้ เทียบระดับไม่ได้", fontsize=11)
ax.legend()
ax.grid(alpha=0.3)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig("fig_train_loss.png", dpi=150, bbox_inches="tight")
plt.show()
print("บันทึกรูปแล้ว: fig_train_loss.png")
print("ถ้าเส้นไหนแบนสนิท: LR ต่ำเกิน หรือ step น้อยเกิน -- เพิ่ม N_TRAIN แล้วรันซ้ำ")


### 8.3 ทางเลือกเสริม (ข้ามได้ทั้งหัวข้อ): GKD แบบ on-policy ด้วย TRL

Logit KD ตามสมการ 3.1 มีจุดอ่อนเชิงโครงสร้าง: นักเรียนเรียนบนประโยคที่ *คนอื่น* เขียน
(teacher forcing) แต่ตอนใช้งานจริงมันต้อง generate ต่อจาก *คำตอบของตัวเอง* —
ความคลาดเคลื่อนสะสมนี้เรียกว่า **exposure bias** และ GKD (สมการ 3.4) แก้ด้วยการให้
นักเรียนสุ่มคำตอบเองระหว่างเทรน แล้วครูให้คะแนนแบบ on-policy

ราคาที่ต้องจ่าย: ช้ากว่า offline KD หลายเท่า (นักเรียนต้อง generate ทุก step
และครูต้องกลับมานั่งบนการ์ดตลอดเวลา) เซลล์ถัดไปจึง **ปิดไว้เป็นค่าเริ่มต้น**
(`RUN_GKD = False`) และถูกออกแบบให้ **ข้ามอย่างสุภาพ** ถ้า TRL รุ่นที่ติดตั้ง
ไม่มี `GKDTrainer` หรือ VRAM ไม่พอ — ผลหลักของตอนนี้ไม่พึ่งหัวข้อนี้


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 8.3 -- GKD (ทางเลือกเสริม): เปิดเมื่อมีเวลาเหลือเท่านั้น (~เพิ่ม 25 นาที)
# -----------------------------------------------------------------------------
RUN_GKD = False     # <- เปลี่ยนเป็น True ถ้าอยากชิมรส on-policy distillation

if not RUN_GKD:
    print("ข้าม GKD (RUN_GKD = False) -- ผลหลักของตอนนี้ครบแล้วโดยไม่ต้องใช้หัวข้อนี้")
else:
    try:
        from datasets import Dataset as HFDataset
        from trl import GKDConfig, GKDTrainer

        # ครูต้องกลับขึ้นการ์ด -- นี่คือราคาแรกของ on-policy
        teacher_gkd = AutoModelForCausalLM.from_pretrained(
            TEACHER_ID,
            torch_dtype=DTYPE,
            attn_implementation=ATTN_IMPL,
            device_map=DEV,
        )
        teacher_gkd.eval()

        student.add_adapter("gkd", make_lora_cfg())
        activate_adapter("gkd")
        gkd_rows = [
            {"messages": [{"role": "user", "content": q},
                          {"role": "assistant", "content": a}]}
            for q, a in train_pairs[:128]
        ]
        gkd_cfg = GKDConfig(
            output_dir="out-gkd",
            beta=0.5,                    # กึ่งกลางเส้น JSD ของสมการ 3.4
            lmbda=0.5,                   # ครึ่งหนึ่งของ batch ใช้คำตอบที่นักเรียนสุ่มเอง
            max_new_tokens=96,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            learning_rate=LR,
            max_steps=40,                # แค่ชิมรส ไม่ใช่การเทรนจริง
            fp16=USE_FP16,
            bf16=SUPPORTS_BF16,
            logging_steps=5,
            save_strategy="no",
            report_to="none",
            seed=SEED,
        )
        gkd_trainer = GKDTrainer(
            model=student,
            teacher_model=teacher_gkd,
            args=gkd_cfg,
            train_dataset=HFDataset.from_list(gkd_rows),
            processing_class=tok_teacher,
        )
        gkd_trainer.train()
        print("GKD จบ -- วัดผลได้ด้วย evaluate(student, ...) แบบเดียวกับ regime อื่น")
        del teacher_gkd
        gc.collect()
        torch.cuda.empty_cache()
    except Exception as exc:
        print(f"GKD รันไม่สำเร็จ -- ข้ามอย่างสุภาพ: {type(exc).__name__}: {exc}")
        print("สาเหตุที่พบบ่อย: TRL รุ่นที่ pin ไว้ไม่มี GKDTrainer หรือ VRAM ไม่พอ")
        print("ผลหลักของตอนนี้ (SeqKD + logit KD) ไม่ได้รับผลกระทบใด ๆ")


---

## 9. เปรียบเทียบ (Comparison)

### gap closed % — ตัวเลขเดียวที่สรุปทั้งตอน

$$
\text{gap closed} = \frac{\text{student}_{\text{after}} - \text{student}_{\text{before}}}{\text{teacher} - \text{student}_{\text{before}}}
$$

ทำไมไม่รายงานคะแนนดิบ — เพราะ "คะแนนขึ้น 4 จุด" ไม่มีความหมายถ้าไม่รู้ว่าช่องว่างทั้งหมด
มีให้ปิดกี่จุด metric นี้ตอบตรงคำถามจริง: **ช่องว่างระหว่างนักเรียนกับครู ปิดไปแล้วกี่เปอร์เซ็นต์**
0% คือไม่ขยับ 100% คือไล่ทันครูพอดี (และถ้าเพดานกับพื้นใกล้กันจนช่องว่างเป็นศูนย์
metric นี้จะนิยามไม่ได้ — โน้ตบุ๊กจะบอกตรง ๆ แทนที่จะหารด้วยศูนย์เงียบ ๆ)

ระวังการอ่านเกินจริง: slice ละ 30 ข้อ ช่วงความเชื่อมั่นกว้างมาก
ให้ดู Wilson CI กับ McNemar ที่ `compare()` พิมพ์ออกมาเสมอ — บนชุดเล็กขนาดนี้
คำตอบที่ซื่อสัตย์มักเป็น "เห็นแนวโน้ม แต่ยังสรุปแน่ชัดไม่ได้"

### สิ่งที่ควรเห็น และวิธีตีความถ้าไม่เห็น

รูปแบบที่ **ควรจะเห็น**: logit KD ≥ SeqKD > base และ tok/sec ก่อน-หลังเท่ากันเป๊ะ

- **SeqKD ชนะ logit KD** → เกิดได้จริงเมื่อเฉลยจริงในชุดข้อมูลเขียนแย่กว่าคำตอบครู
  (logit KD ของเราเทรนบนเฉลยจริง) — นี่คือข้อมูล ไม่ใช่ความล้มเหลว รายงานมันตรง ๆ
- **ทุกแถวแทบไม่ขยับ** → 384 ตัวอย่าง 24 step คือขนาดสาธิตกลไก ดู loss curve ก่อนสรุป
  แล้วเพิ่ม `N_TRAIN` — ช่องว่าง 1.7B→0.6B ก็แคบกว่างานกลั่นจริง (70B→7B) มาก
- **th_ratio ตก** → นักเรียนรับนิสัยหลุดภาษาอังกฤษมาจากครู — ดูตัวเลข th_ratio
  ของคำตอบครูที่พิมพ์ไว้ในหัวข้อ 6.3 ประกอบ

### แล้ววัดนิสัยที่ติดมาจากครูด้วย TH-SAFE

การกลั่นถ่ายทอด **ทุกอย่าง** รวมทั้งอคติและความผิดพลาดของครู นักเรียนไม่มีกลไกแยกแยะว่า
ส่วนไหนของ distribution คือความรู้ ส่วนไหนคือนิสัยเสีย เราจึงวัดนักเรียนหลังกลั่นบน
**TH-SAFE** (slice ความปลอดภัยของ KobEval-TH: ครึ่ง unsafe ต้องปฏิเสธ /
ครึ่ง benign ต้องตอบ) แล้วรายงานตามจริง — ตัวเลขบอกว่ารับอะไรติดมา ก็เขียนลงรายงาน
ไม่ใช่ลบแถวนั้นทิ้ง


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 9.1 -- วัด KobEval-TH ซ้ำด้วยสัญญาเดิมทุกตัวอักษร: นักเรียนทั้งสอง regime
# -----------------------------------------------------------------------------
def eval_student(tag, name, out_path):
    activate_adapter(tag)
    student.eval()
    t0 = time.time()
    rep = evaluate(
        student,
        tokenizer,
        slices=KOBEVAL_SLICES,
        seed=42,
        model_name=name,
        out_path=out_path,
    )
    print(f"[{tag}] วัดเสร็จใน {time.time() - t0:.0f} วินาที\n")
    return rep


after_sft = eval_student("sft", "0.6B + SFT เฉลยจริง (control)", "results_after_sft.json")
after_seqkd = eval_student("seqkd", "0.6B + SeqKD", "results_after_seqkd.json")
after_kd = eval_student("kd", f"0.6B + logit KD (T={T_KD:g}, α={ALPHA:g})",
                        "results_after_kd.json")

print(f"{'โมเดล':28s} {'TH-INSTR':>9s} {'95% CI':>16s} {'th_ratio':>9s}")
print("-" * 68)
for rep in (baseline, after_sft, after_seqkd, after_kd, teacher_report):
    s = rep["slices"]["TH-INSTR"]
    ci = f"[{s['ci_low']:.2f}-{s['ci_high']:.2f}]"
    print(f"{rep['model'][:28]:28s} {s['accuracy']:9.3f} {ci:>16s} {s['th_ratio']:9.2f}")


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 9.2 -- gap closed % + กราฟ + McNemar
# -----------------------------------------------------------------------------
def gap_closed(after_acc):
    gap = ACC_CEIL - ACC_FLOOR
    if gap <= 1e-9:
        return None      # เพดานไม่สูงกว่าพื้น -- metric นี้นิยามไม่ได้ ห้ามหารด้วยศูนย์เงียบ ๆ
    return (after_acc - ACC_FLOOR) / gap


GAP_CLOSED = {
    "sft": gap_closed(after_sft["slices"]["TH-INSTR"]["accuracy"]),
    "seqkd": gap_closed(after_seqkd["slices"]["TH-INSTR"]["accuracy"]),
    "kd": gap_closed(after_kd["slices"]["TH-INSTR"]["accuracy"]),
}

print(f"พื้น (นักเรียน)  : {100 * ACC_FLOOR:.1f}%   เพดาน (ครู): {100 * ACC_CEIL:.1f}%")
for tag, gc_val in GAP_CLOSED.items():
    if gc_val is None:
        print(f"  {tag:6s}: gap closed นิยามไม่ได้ (ครูไม่ได้สูงกว่านักเรียนบน slice นี้)")
    else:
        print(f"  {tag:6s}: gap closed = {100 * gc_val:+.1f}%")

use_thai_font()
if all(v is not None for v in GAP_CLOSED.values()):
    fig, ax = plt.subplots(figsize=(8.0, 3.6))
    names = ["นักเรียน (base)", "SFT เฉลยจริง (control)", "SeqKD", f"logit KD (T={T_KD:g})"]
    vals = [0.0, 100 * GAP_CLOSED["sft"], 100 * GAP_CLOSED["seqkd"], 100 * GAP_CLOSED["kd"]]
    colors = ["#94a3b8", "#f59e0b", "#8b5cf6", "#2563eb"]
    ax.barh(names, vals, color=colors, height=0.55)
    for y, v in enumerate(vals):
        ax.text(v + (1.5 if v >= 0 else -1.5), y, f"{v:+.1f}%",
                va="center", ha="left" if v >= 0 else "right", fontsize=10)
    ax.axvline(0, color="#334155", lw=1)
    ax.axvline(100, color="#16a34a", lw=1.5, ls="--")
    ax.text(100, -0.45, "ครู = 100%", color="#16a34a", fontsize=9, ha="center")
    ax.set_xlabel("gap closed (%) -- 0% = นักเรียนก่อนกลั่น, 100% = ไล่ทันครู")
    ax.set_title("ปิดช่องว่างครู-นักเรียนไปแล้วกี่เปอร์เซ็นต์ (ที่ tok/sec เท่าเดิมเป๊ะ)",
                 fontsize=11)
    ax.grid(axis="x", alpha=0.3)
    ax.set_axisbelow(True)
    fig.tight_layout()
    fig.savefig("fig_gap_closed.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("บันทึกรูปแล้ว: fig_gap_closed.png")

# ตารางมาตรฐานของซีรีส์ + McNemar (นักเรียนก่อน vs หลัง logit KD)
table = compare(baseline, after_kd, out_path="results_table.json")

plot_before_after(
    baseline,
    after_kd,
    slices=KOBEVAL_SLICES,
    title=f"ตอนที่ 7: นักเรียนก่อน vs หลัง logit KD (T={T_KD:g}, α={ALPHA:g})",
    out_path="fig_before_after.png",
    results_json="results_before_after.json",
)


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 9.3 -- TH-SAFE บนนักเรียนหลังกลั่น: นิสัยของครูติดมาด้วยไหม
# -----------------------------------------------------------------------------
activate_adapter("kd")
student.eval()
t0 = time.time()
thsafe_kd = evaluate(
    student,
    tokenizer,
    slices=["TH-SAFE"],
    seed=42,
    model_name=f"0.6B + logit KD (T={T_KD:g})",
    out_path="results_thsafe_kd.json",
)
print(f"วัด TH-SAFE เสร็จใน {time.time() - t0:.0f} วินาที\n")

s = thsafe_kd["slices"]["TH-SAFE"]
print(f"TH-SAFE acc รวม            : {s['accuracy']:.3f} "
      f"[95% CI {s['ci_low']:.3f}-{s['ci_high']:.3f}]")
print(f"unsafe_compliance_rate      : {s['unsafe_compliance_rate']:.3f} "
      f"(สัดส่วนคำขออันตรายที่ 'ยอมตอบ' -- ต่ำ = ดี)")
print(f"over_refusal_rate           : {s['over_refusal_rate']:.3f} "
      f"(สัดส่วนคำถามบริสุทธิ์ที่ 'ปฏิเสธเกินเหตุ' -- ต่ำ = ดี)")
print()
print("อ่านผลอย่างซื่อสัตย์:")
print("- นักเรียนเป็นโมเดล base ที่ 'ไม่เคยถูกสอนให้ปฏิเสธ' มาก่อน พฤติกรรมปลอดภัย")
print("  เกือบทั้งหมดที่เห็น (ถ้ามี) จึงรับมาจาก distribution/คำตอบของครูระหว่างกลั่น")
print("- ถ้า unsafe_compliance สูง: การกลั่นไม่ได้สร้างรั้วให้ฟรี ๆ -- ตอนที่ 8 (Guardrails)")
print("  มีอยู่เพราะเหตุนี้ อย่า deploy นักเรียนกลั่นโดยไม่มีรั้วภายนอก")
print("- ถ้าอยากได้แถวครูมาเทียบ: รัน evaluate(teacher, ...) บน TH-SAFE เพิ่มได้")
print("  (~4 นาที -- ต้องโหลดครูกลับขึ้นการ์ดก่อน)")


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 9.4 -- tok/sec ก่อน/หลัง (ต้องเท่ากัน) + export samples.json
# (วิดเจ็ต BeforeAfterCompare บนบล็อกอ่านไฟล์นี้ไปแสดงคำตอบจริง ไม่ใช่คำตอบที่แต่งขึ้น)
# -----------------------------------------------------------------------------
def generate_probe(mode, max_new_tokens=96):
    """generate บน probe ทั้งชุด -- คืน (ข้อความ, tok/sec)"""
    prompts = [PROMPT_TMPL.format(q=q) for q in probe_questions]
    tokenizer.padding_side = "left"
    try:
        enc = tokenizer(prompts, return_tensors="pt", padding=True,
                        truncation=True, max_length=MAX_PROMPT_LEN).to(DEV)
        torch.manual_seed(SEED)
        torch.cuda.synchronize()
        t0 = time.time()
        if mode == "base":
            with torch.no_grad(), student.disable_adapter():
                out = student.generate(**enc, max_new_tokens=max_new_tokens,
                                       do_sample=False,
                                       pad_token_id=tokenizer.pad_token_id)
        else:
            activate_adapter(mode)
            student.eval()
            with torch.no_grad():
                out = student.generate(**enc, max_new_tokens=max_new_tokens,
                                       do_sample=False,
                                       pad_token_id=tokenizer.pad_token_id)
        torch.cuda.synchronize()
        dt = time.time() - t0
        new = out[:, enc["input_ids"].shape[1]:]
        n_new = int((new != tokenizer.pad_token_id).sum())
        texts = tokenizer.batch_decode(new, skip_special_tokens=True)
    finally:
        tokenizer.padding_side = "right"
    return [t.strip() for t in texts], n_new / dt


base_texts, tps_base = generate_probe("base")
seqkd_texts, _ = generate_probe("seqkd")
kd_texts, tps_kd = generate_probe("kd")

TOKS_PER_SEC = {"before": tps_base, "after_kd": tps_kd}
print(f"tok/sec: adapter ปิด (= ความเร็วหลัง merge/deploy) = {tps_base:.1f}")
print(f"tok/sec: adapter เปิด (ยังมีภาระ BAx ที่ยังไม่ merge)  = {tps_kd:.1f}")
print("ส่วนต่างคือ 'ภาษีของ adapter ที่ยังไม่ merge' ไม่ใช่ต้นทุนของ distillation")
print("distillation เปลี่ยน 'น้ำหนัก' ไม่เปลี่ยน 'สถาปัตยกรรม' -- merge แล้ว (W' = W + (α/r)BA, ตอนที่ 10)")
print("นักเรียนกลั่นกลับไปวิ่งที่ความเร็วฐานเป๊ะ: นี่คือประเด็นของทั้งตอน -- คุณภาพขยับเข้าหาครู latency ไม่ขยับ")

sample_items = []
for i, q in enumerate(probe_questions):
    outputs = {
        "base": base_texts[i],
        "seqkd": seqkd_texts[i],
        "kd": kd_texts[i],
        "teacher": teacher_probe_answers.get(q, ""),
    }
    metrics = {}
    for key, text in outputs.items():
        metrics[key] = {
            "tokens": len(tokenizer(text, add_special_tokens=False).input_ids),
            "thai_ratio": round(th_ratio(text), 4),
        }
    sample_items.append({"prompt": q, "outputs": outputs, "metrics": metrics})

with open("samples.json", "w", encoding="utf-8") as f:
    json.dump({"items": sample_items}, f, ensure_ascii=False, indent=2)
print(f"\nเขียน samples.json แล้ว ({len(sample_items)} คำถาม x 4 ระบบ)")
print("\nตัวอย่างคำตอบเดียวกันสามระบบ:")
print("  base :", base_texts[0][:130].replace(chr(10), " "))
print("  kd   :", kd_texts[0][:130].replace(chr(10), " "))
print("  ครู  :", teacher_probe_answers.get(probe_questions[0], "")[:130].replace(chr(10), " "))


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 9.5 -- รวมทุกตัวเลขลง results.json ไฟล์เดียว เพื่อให้วิดเจ็ตบนบล็อกอ่านค่าจริง
# -----------------------------------------------------------------------------
from kobeval import write_results


def kobeval_summary(rep):
    return {
        "model": rep["model"],
        "accuracy": {k: v["accuracy"] for k, v in rep["slices"].items()},
        "ci": {k: [v["ci_low"], v["ci_high"]] for k, v in rep["slices"].items()},
        "th_ratio": rep["overall"]["th_ratio"],
    }


payload = {
    "post": 7,
    "slug": "llm-07-model-distillation",
    "notebook": "07_model_distillation.ipynb",
    "teacher": TEACHER_ID,
    "student": STUDENT_ID,
    "gpu": GPU_NAME,
    "supports_bf16": SUPPORTS_BF16,
    "eval_contract": EVAL_CONTRACT,
    "datasets": ["airesearch/wangchanx-seed-free-synthetic-instruct-thai-120k"],
    "vocab": {"vocab_size": VOCAB_SIZE, "shared": True},
    "vram": {"both_models_gb": VRAM_BOTH_GB, "after_teacher_freed_gb": VRAM_AFTER_FREE},
    "data": {
        "n_prompts_pool": N_PROMPTS,
        "n_train": len(gold_train),
        "n_probe": len(probe_questions),
        "max_len": MAX_LEN,
        "mean_gold_th_ratio": mean_ans_th,
        "mean_teacher_answer_th_ratio": mean_teacher_th,
    },
    "kd": {
        "temperature": T_KD,
        "alpha": ALPHA,
        "top_k": TOP_K,
        "coverage_at_T2": coverage_samples[0],
        "float64_check_abs_diff": diff,
        "t_sweep": {"temperatures": TS, "grad_raw": grad_raw, "grad_fixed": grad_fixed,
                    "loglog_slope": slope},
    },
    "memory_math": {
        "full_vocab_batch_bytes": full_batch,
        "topk_batch_bytes": topk_batch,
        "ratio": full_batch / topk_batch,
        "full_vocab_offline_gb": full_offline / 1e9,
        "topk_offline_gb": topk_offline / 1e9,
    },
    "hyperparameters": {
        "learning_rate": LR,
        "per_device_train_batch_size": BATCH_SIZE,
        "gradient_accumulation_steps": GRAD_ACCUM,
        "num_train_epochs": 1,
        "lora_r": 16,
        "lora_alpha": 32,
        "fp16": USE_FP16,
        "gen_max_new_tokens": GEN_MAX_NEW,
    },
    "training": {
        "seqkd": {k: v for k, v in SEQKD_RUN.items() if k != "log_history"},
        "kd": {k: v for k, v in KD_RUN.items() if k != "log_history"},
        "log_history": {"seqkd": SEQKD_RUN["log_history"], "kd": KD_RUN["log_history"]},
    },
    "tokens_per_sec": TOKS_PER_SEC,
    "gap_closed": GAP_CLOSED,
    "kobeval": {
        "slices_run": KOBEVAL_SLICES,
        "student_before": kobeval_summary(baseline),
        "teacher": kobeval_summary(teacher_report),
        "after_sft_control": kobeval_summary(after_sft),
        "after_seqkd": kobeval_summary(after_seqkd),
        "after_kd": kobeval_summary(after_kd),
        "mcnemar": table.get("mcnemar"),
        "markdown_table": table["markdown"],
    },
    "th_safe_after_kd": {
        "accuracy": thsafe_kd["slices"]["TH-SAFE"]["accuracy"],
        "ci": [thsafe_kd["slices"]["TH-SAFE"]["ci_low"],
               thsafe_kd["slices"]["TH-SAFE"]["ci_high"]],
        "unsafe_compliance_rate": thsafe_kd["slices"]["TH-SAFE"]["unsafe_compliance_rate"],
        "over_refusal_rate": thsafe_kd["slices"]["TH-SAFE"]["over_refusal_rate"],
    },
    "samples_file": "samples.json",
    "tokens_file": "tokens.json",
    "figures": [
        "fig_temperature_softens.png",
        "fig_t2_gradient.png",
        "fig_train_loss.png",
        "fig_gap_closed.png",
        "fig_before_after.png",
    ],
}

path = write_results(payload, "results.json")
print(f"เขียนไฟล์แล้ว: {path}")
print(json.dumps({
    "gap_closed": payload["gap_closed"],
    "tokens_per_sec": payload["tokens_per_sec"],
    "th_safe_after_kd": payload["th_safe_after_kd"],
}, ensure_ascii=False, indent=2))


---

## 10. สรุป (Summary)

- **Dark knowledge อยู่ในคำตอบที่ผิดของครู** — การจัดอันดับเหนือทางเลือกผิด ๆ เข้ารหัส
  โครงสร้างความคล้ายที่ hard label ไม่มีวันบอก และ **temperature คือปุ่มที่เปิดเผยมัน**
  (เราดูจาก logits จริงของครูแล้วในรูปแรก — ข้อมูลเท่าเดิม การมองเห็นต่างกัน)
- **ตัวคูณ $T^2$ ไม่ใช่เครื่องราง** — gradient ของ soft loss สเกลตาม $1/T^2$
  และเราวัดมันจริงด้วย autograd: slope log-log ≈ −2 เป๊ะตามที่อนุมาน
  คูณคืนเพื่อให้จูน T ได้โดยไม่แอบเปลี่ยน learning rate ของตัวเอง
- **KD loss ที่เขียนเอง ตรงกับ float64 อิสระถึง 1e-6** — masking, การ shift ตำแหน่ง
  และ renormalize บน top-64 ถูกพิสูจน์ ไม่ใช่ขอให้เชื่อ
- **SeqKD คือ SFT บนคำตอบครู** — baseline ที่ถูกที่สุด และเป็นทางเดียวเมื่อ tokenizer ไม่ตรงกัน
- **Logit KD ต้องการ vocab เดียวกัน** — `assert` ก่อนเสมอ เพราะ KL เทียบมิติต่อมิติ
- **Top-64 คือวิศวกรรม ไม่ใช่ทฤษฎี** — logits เต็ม vocab หนึ่ง batch คือ tensor ~622 MB
  เก็บ 64 อันดับแรกเหลือไม่ถึง 1 MB โดยเสีย coverage ไม่ถึง 1% (เราวัด coverage จริงแล้ว)
- **gap closed คือ metric ที่ตอบคำถามจริง** — ปิดช่องว่างครู-นักเรียนไปกี่เปอร์เซ็นต์
  ที่ tok/sec เท่าเดิมเป๊ะ — "ได้คุณภาพ (บางส่วนของ) ครู ที่ราคาของนักเรียน"
- **การกลั่นถ่ายทอดทุกอย่าง รวมทั้งนิสัยเสียของครู** — เราวัด TH-SAFE บนนักเรียนหลังกลั่น
  แล้วรายงานตามจริง โมเดลที่รับทุกอย่างมาโดยไม่ตั้งคำถาม ต้องมีรั้วของตัวเอง

**ตอนต่อไป:** Guardrails — นักเรียนของเราเพิ่งรับทั้งความรู้และนิสัยของครูมาเต็ม ๆ
ตอนหน้าเราจะสร้างรั้วรอบโมเดล: จับ input อันตรายก่อนถึงตัวโมเดล
จับ output อันตรายก่อนถึงผู้ใช้ และวัด trade-off ระหว่างความปลอดภัยกับความน่าใช้
ด้วยตัวเลขจริง


---

## ข้อจำกัดของการทดลองนี้

เขียนไว้ตรง ๆ เพื่อไม่ให้ตัวเลขข้างบนถูกอ่านเกินกว่าที่มันบอกได้จริง

1. **ช่องว่าง 1.7B → 0.6B เป็นช่องว่างที่แคบ** — ครูของเราไม่ได้เก่งกว่านักเรียนแบบทิ้งห่าง
   กำไรที่วัดได้จึงย่อมแคบตาม อย่าเอาตัวเลข gap closed จากคู่นี้ไปเทียบกับงานกลั่น
   70B → 7B ที่ช่องว่างกว้างกว่าหลายเท่า การทดลองนี้พิสูจน์ **กลไกและวิธีวัด**
   ไม่ใช่ตัวเลขสุดท้าย

2. **384 ตัวอย่าง 24 step คือขนาดสาธิต** — งานกลั่นจริงใช้ข้อมูลระดับแสนถึงล้านตัวอย่าง
   ต่างกันหลาย order of magnitude ถ้าผลของคุณแทบไม่ขยับ ให้เพิ่ม `N_TRAIN` ก่อนสรุป

3. **ขนาดชุดวัดผลเล็กมาก** — KobEval-TH มี slice ละ 30 ข้อ ช่วงความเชื่อมั่นจึงกว้าง
   ความต่างระดับ 1-2 ข้อ **ไม่ใช่** ความต่างที่มีนัยสำคัญ ให้ดูค่า p จาก McNemar
   ที่ `compare()` พิมพ์ออกมาด้วยเสมอ gap closed % บนตัวเศษ/ตัวส่วนเล็ก ๆ
   จึงเหวี่ยงแรงมาก — อ่านมันคู่กับ CI เสมอ

4. **ครูที่ใหญ่กว่านี้ไม่พอดีกับ T4** — 7B fp16 กิน ~14 GB แค่ตัวเดียวก็เกือบเต็มการ์ด
   ทางออกคือสิ่งที่เราฝึกทำในตอนนี้พอดี: precompute top-K logits แบบ offline
   บนเครื่องเช่ารายชั่วโมงครั้งเดียว แล้วเทรนนักเรียนที่ไหนก็ได้จากไฟล์ไม่กี่ร้อย MB
   โครงสร้าง offline ที่ดูเป็นการประนีประนอมบน Colab แท้จริงคือวิธีที่งานสเกลจริงเขาทำกัน

5. **top-64 คือการประมาณ** — เราวัด coverage จริง (~99%+ ที่ T=2) แต่ที่ T สูงกว่านี้
   มวลจะไหลลงหางมากขึ้น และ K=64 อาจไม่พอ ถ้าจะกวาด T ให้กวาด K ไปด้วย

6. **การกลั่นถ่ายทอดอคติของครู และ TH-SAFE วัดได้แค่บางมุม** — slice ละ 30 ข้อ
   ตรวจการปฏิเสธด้วย keyword ซึ่งหยาบกว่าการให้มนุษย์ตรวจ ผล TH-SAFE ในตอนนี้
   เป็นสัญญาณเตือน ไม่ใช่ใบรับรองความปลอดภัย

7. **รันครั้งเดียว ไม่ได้ทำ multiple seeds** — ใช้ greedy decoding และ seed=42 ตายตัว
   เพื่อให้ทำซ้ำได้ แต่ไม่ได้รายงานความแปรปรวนจากการเทรนหลาย seed

8. **ฮาร์ดแวร์จำกัด** — ทุกอย่างถูกบีบให้รันได้บน T4 ฟรี ซึ่งแปลว่าต้องใช้ fp16 แทน bf16,
   ใช้ sdpa แทน FlashAttention-2, batch เล็ก และ sequence สั้น
   ผลบนฮาร์ดแวร์ที่ใหญ่กว่าอาจต่างออกไป

---

*ซีรีส์นี้เผยแพร่ภายใต้สัญญาอนุญาต [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/) — ใช้ต่อได้โดยอ้างอิงที่มา ห้ามใช้เชิงพาณิชย์ และต้องเผยแพร่ต่อด้วยสัญญาเดียวกัน — [kobkrit.com](https://kobkrit.com)*
